In [1]:
import re
import json


def extract_citations(text):
    # More specific regex pattern with exact start and end
    pattern = r'ITEM CSL_CITATION .*?"schema":"https://github\.com/citation-style-language/schema/raw/master/csl-citation\.json"}'

    # Dictionary to store id and citation-key mappings
    citation_mapping = {}

    # Find all matches in the text
    matches = re.finditer(pattern, text, re.DOTALL)

    for match in matches:
        # Extract the JSON part (removing "ITEM CSL_CITATION " prefix)
        json_str = match.group(0)[17:]  # 17 is the length of "ITEM CSL_CITATION "

        try:
            # Parse the JSON content of the citation
            citation_data = json.loads(json_str)

            # Extract id and citation-key from the citation items
            for item in citation_data["citationItems"]:
                if "id" in item and "citation-key" in item["itemData"]:
                    citation_id = item["id"]
                    citation_key = item["itemData"]["citation-key"]
                    citation_mapping[citation_id] = citation_key
        except json.JSONDecodeError:
            print(f"Failed to parse JSON in citation: {json_str[:100]}...")
        except KeyError:
            print(f"Missing required keys in citation data")

    return citation_mapping


# Example usage:
with open("diss_24_10_30.txt", "r", encoding="utf-8") as file:
    text_content = file.read()

citations = extract_citations(text_content)
print("Citation ID to Citation Key mapping:")
for citation_id, citation_key in citations.items():
    print(f"ID: {citation_id} -> Key: {citation_key}")

Citation ID to Citation Key mapping:
ID: 1034 -> Key: fillmoreAlternativeChecklistTheories1975
ID: 358 -> Key: fillmoreFrameSemantics1982
ID: 266 -> Key: busseFrameSemantikKompendium2012
ID: 1394 -> Key: busseDiskurslinguistikAlsEpistemologie2008
ID: 133 -> Key: firthSynopsisLinguisticTheory1957
ID: 865 -> Key: harrisDistributionalStructure1954
ID: 128 -> Key: sinclairCorpusConcordanceCollocation1991
ID: 1040 -> Key: perkuhnKorpuslinguistikUnbekannteWesen2006
ID: 249 -> Key: tognini-bonelliCorpusLinguisticsWork2001
ID: 106 -> Key: bubenhoferSprachgebrauchsmusterKorpuslinguistikAls2009
ID: 111 -> Key: feilkeTextroutineTextsemantikUnd2003
ID: 1333 -> Key: feilkeOberflaecheUndPerformanz2009
ID: 263 -> Key: linkeBegriffsgeschichteDiskursgeschichteSprachgebrauchsgeschichte2003
ID: 1002 -> Key: busseDiskursSpracheGesellschaftliches2013
ID: 225 -> Key: scharlothWuchernRhizomeLinguistische2013
ID: 313 -> Key: ackermannMetamorphosenExtremismusbegriffesDiskursanalytische2015
ID: 293 -> Key: czul

In [2]:
def replace_citations_in_tex(tex_file_path, citation_mapping):
    # Read the tex file
    with open(tex_file_path, "r", encoding="utf-8") as file:
        content = file.readlines()  # Read line by line

    new_content = []

    for line in content:
        if "\\autocite" in line:
            # Pattern to find one or more digits (possibly separated by commas) within curly braces
            pattern = r"\{(\d+(?:\s*,\s*\d+)*)\}"

            def replace_citation(match):
                cite_string = match.group(1)  # The string of citation IDs
                # Split the string into individual citation IDs
                cite_ids = [int(id.strip()) for id in cite_string.split(",")]

                # Map each citation ID to its new value
                mapped_citations = []
                for cite_id in cite_ids:
                    if cite_id in citation_mapping:
                        mapped_citations.append(citation_mapping[cite_id])
                    else:
                        print(f"Warning: Citation ID {cite_id} not found in mapping")
                        mapped_citations.append(str(cite_id))

                # Join the mapped citations with commas
                return "{" + ", ".join(mapped_citations) + "}"

            # Replace all citation IDs in this line
            modified_line = re.sub(pattern, replace_citation, line)
            new_content.append(modified_line)
        else:
            new_content.append(line)

    # Write the modified content to a new file
    output_file_path = tex_file_path.replace(".tex", "_citekeys.tex")
    with open(output_file_path, "w", encoding="utf-8") as file:
        file.writelines(new_content)

    return output_file_path


# Example usage:
tex_file_path = "output.tex"

output_citekeys = replace_citations_in_tex(tex_file_path, citations)
print(f"Modified file saved as: {output_citekeys}")

Modified file saved as: output_citekeys.tex


In [3]:
with open(output_citekeys, "r", encoding="utf-8") as file:
    output_citekeys = file.read()

In [4]:
import re


def split_sections(text):
    # Pattern to match \section{name}\label{label} format
    # This will capture the section name in group 1
    pattern = r"\\section\{([^}]*)\}\\label\{[^}]*\}"

    # Split the text using the pattern as delimiter, but keep the delimiters
    # The (?=...) is a positive lookahead that keeps the delimiter in the result
    sections = re.split(f"(?={pattern})", text)

    # Remove empty sections (if any)
    sections = [s.strip() for s in sections if s.strip()]

    # Create dictionary to store results
    section_dict = {}

    for section in sections:
        # Find the section name using the pattern
        match = re.search(pattern, section)
        if match:
            # Get the section name from the first capture group
            section_name = match.group(1)
            # Store the entire section content with the section name as key
            if section_name == "Daten und Methodik":
                section_dict["methodik"] = section
            elif section_name == "Diskussion, Limitationen, Perspektiven und Fazit":
                section_dict["diskussion"] = section
            elif section_name == "Literatur-, Abbildungs- und Tabellenverzeichnis":
                section_dict["verzeichnisse"] = section
            else:
                section_dict[section_name.lower()] = section

    return section_dict

In [5]:
tables = {
    1: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.3022}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.3206}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.3772}}@{}}
\toprule\noalign{}
\multicolumn{3}{@{}>{\centering\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{1.0000} + 4\tabcolsep}@{}}{%
\begin{minipage}[b]{\linewidth}\centering
nächste Nachbarn (Cosinus-Ähnlichkeit)
\end{minipage}} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\emph{Junge} & \emph{Alternative} & \emph{Junge\_Alternative} \\
\emph{junger\_Mann} (0.71)

\emph{Mann} (0.66)

\emph{Hund} (0.66)

\emph{Kleinkind} (0.65)

\emph{Bub} (0.65)

\emph{elfjährige} (0.64)

\emph{Säugling} (0.64)

\emph{Teenager} (0.64)

\emph{Baby} (0.63)

\emph{Senior} (0.63) & \emph{Ergänzung} (0.67)

\emph{Alternativen} (0.66)

\emph{Option} (0.65)

\emph{echte\_Alternative} (0.62)

\emph{bessere\_Alternative} (0.56)

\emph{Lösung} (0.55)

\emph{Blaupause} (0.54)

\emph{sinnvolle\_Ergänzung} (0.52)

\emph{Parallele} (0.52)

\emph{Möglichkeit} (0.52) & \emph{jA} (0.88)

\emph{Verfassungsschutz\_stuft} (0.8)

\emph{AfD-Flügel} (0.77)

\emph{Nachwuchsorganisation} (0.75)

\emph{Beobachtungsfall} (0.73)

\emph{Gesamtpartei} (0.72)

\emph{rechtsextrem} (0.71)

\emph{rechtsextremen\_Verdachtsfall} (0.7)

\emph{Beobachtungsobjekt} (0.68)

\emph{Identitäre\_Bewegung} (0.68) \\
\end{longtable}

% \phantomsection\label{_Ref162017554}{}Tabelle : Nächste Nachbarn im Word-Embedding-Modell 2020 -- 2021 zu \emph{Junge}, \emph{Alternative} und \emph{Junge\_Alternative} mit Cosinus-Ähnlichkeit.
""",
        r"""
\begin{table}
  \caption{Nächste Nachbarn im Word-Embedding-Modell 2020 -- 2021 zu \emph{Junge}, \emph{Alternative} und \emph{Junge\_Alternative} mit Cosinus-Ähnlichkeit.}
  \label{tab:phrasing}
   \begin{tabularx}{\textwidth}{X X X}
    \lsptoprule
              \emph{Junge}                    & \emph{Alternative}             & \emph{Junge\_Alternative} \\ % Header row
    \midrule
    \emph{junger\_Mann} (0.71)      & \emph{Ergänzung} (0.67)        & \emph{jA} (0.88)          \\
    \emph{Mann} (0.66)              & \emph{Alternativen} (0.66)     & \emph{Verfassungsschutz\_stuft} (0.80) \\
    \emph{Hund} (0.66)              & \emph{Option} (0.65)           & \emph{AfD-Flügel} (0.77)  \\
    \emph{Kleinkind} (0.65)         & \emph{echte\_Alternative} (0.62) & \emph{Nachwuchsorganisation} (0.75) \\
    \emph{Bub} (0.65)               & \emph{bessere\_Alternative} (0.56) & \emph{Beobachtungsfall} (0.73) \\
    \emph{elfjährige} (0.64)        & \emph{Lösung} (0.55)           & \emph{Gesamtpartei} (0.72) \\
    \emph{Säugling} (0.64)          & \emph{Blaupause} (0.54)        & \emph{rechtsextrem} (0.71) \\
    \emph{Teenager} (0.64)          & \emph{sinnvolle\_Ergänzung} (0.52) & \emph{rechtsextremen\_Verdachtsfall} (0.70) \\
    \emph{Baby} (0.63)              & \emph{Parallele} (0.52)        & \emph{Beobachtungsobjekt} (0.68) \\
    \emph{Senior} (0.63)            & \emph{Möglichkeit} (0.52)      & \emph{Identitäre\_Bewegung} (0.68) \\
    \lspbottomrule
   \end{tabularx}
  \end{table}
""",
    ),
    2: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.3333}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.3333}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.3333}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Subkorpus
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Types
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Types \textgreater= 25
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
1999 -- 2001 & 2 541 673 & 132 041 \\
2002 -- 2004 & 2 759 025 & 148 597 \\
2005 -- 2007 & 2 994 777 & 168 204 \\
2008 -- 2010 & 3 048 237 & 183 405 \\
2011 -- 2013 & 3 196 110 & 203 417 \\
2014 -- 2016 & 3 163 474 & 201 286 \\
2017 -- 2019 & 2 910 547 & 190 915 \\
2020 -- 2021 & 1 864 503 & 110 389 \\
\textsc{Taz} & 5 789 089 & 365 815 \\
\textsc{Spiegel} & 4 883 715 & 342 740 \\
\textsc{Welt} & 5 002 421 & 355 619 \\
\end{longtable}

% \phantomsection\label{_Ref162017306}{}Tabelle : Type-Anzahlen und Type-Anzahlen mit einer Mindestfrequenz von 25 pro Subkorpus.""",
        r"""\begin{table}
  \caption{Type-Anzahlen und Type-Anzahlen mit einer Mindestfrequenz von 25 pro Subkorpus.}
  \label{tab:type_frequencies}
   \begin{tabularx}{\textwidth}{X X X}
    \lsptoprule
    Subkorpus     & Types      & Types $\geq$ 25\\ %table header
    \midrule
    1999 -- 2001  & 2541673  & 132041 \\
    2002 -- 2004  & 2759025  & 148597 \\
    2005 -- 2007  & 2994777  & 168204 \\
    2008 -- 2010  & 3048237  & 183405 \\
    2011 -- 2013  & 3196110  & 203417 \\
    2014 -- 2016  & 3163474  & 201286 \\
    2017 -- 2019  & 2910547  & 190915 \\
    2020 -- 2021  & 1864503  & 110389 \\
    \textsc{Taz}  & 5789089  & 365815 \\
    \textsc{Spiegel} & 4883715  & 342740 \\
    \textsc{Welt} & 5002421  & 355619 \\
    \lspbottomrule
   \end{tabularx}
  \end{table}""",
    ),
    3: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 10\tabcolsep) * \real{0.2963}}
  >{\raggedright\arraybackslash}p{(\linewidth - 10\tabcolsep) * \real{0.1419}}
  >{\raggedright\arraybackslash}p{(\linewidth - 10\tabcolsep) * \real{0.1407}}
  >{\raggedright\arraybackslash}p{(\linewidth - 10\tabcolsep) * \real{0.1250}}
  >{\raggedright\arraybackslash}p{(\linewidth - 10\tabcolsep) * \real{0.1610}}
  >{\raggedright\arraybackslash}p{(\linewidth - 10\tabcolsep) * \real{0.1352}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
word
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
count\_coi
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
count\_ref
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
exp\_coi
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
exp\_ref
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
ll
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\emph{Anschlag} & 358 & 3179 & 1,42 & 3535,58 & 3288,71 \\
\emph{Halle} & 245 & 5235 & 2,20 & 5477,80 & 1837,30 \\
\emph{in} & 1856 & 1749678 & 701,88 & 1750832,12 & 1344,40 \\
\emph{Opfer} & 227 & 11983 & 4,89 & 12205,11 & 1303,45 \\
\emph{von} & 1122 & 837359 & 336 & 838145 & 1153,76 \\
\emph{rassistischen\_Anschlag} & 78 & 15 & 0,04 & 92,96 & 1138,29 \\
\emph{nach} & 681 & 335175 & 134,59 & 335721,41 & 1125,60 \\
\emph{neun} & 172 & 9137 & 3,73 & 9305,27 & 985,25 \\
\emph{Hanau} & 100 & 3160 & 1,31 & 3258,69 & 673,52 \\
\emph{Anschlags} & 60 & 409 & 0,19 & 468,71 & 580,38 \\
\end{longtable}

% \phantomsection\label{_Ref162017574}{}Tabelle : Beispiel-Output einer Kollokationsanalyse mit \emph{polmineR} zu \emph{Hanau} im Subkorpus 2020 -- 2021.""",
        r"""\begin{table}
\caption{Beispiel-Output einer Kollokationsanalyse mit \emph{polmineR} zu \emph{Hanau} im Subkorpus 2020 -- 2021.}
\label{tab:collocation_analysis}
 \begin{tabularx}{\textwidth}{>{\hsize=1.2\hsize}X >{\hsize=0.96\hsize}X >{\hsize=0.96\hsize}X >{\hsize=0.96\hsize}X >{\hsize=0.96\hsize}X >{\hsize=0.96\hsize}X}
  \lsptoprule
  word & count\_coi & count\_ref & exp\_coi & exp\_ref & ll \\
  \midrule
  \emph{Anschlag}              & 358   & 3179    & 1.42   & 3535.58   & 3288.71 \\
  \emph{Halle}                 & 245   & 5235    & 2.20   & 5477.80   & 1837.30 \\
  \emph{in}                    & 1856  & 1749678 & 701.88 & 1750832.12 & 1344.40 \\
  \emph{Opfer}                 & 227   & 11983   & 4.89   & 12205.11  & 1303.45 \\
  \emph{von}                   & 1122  & 837359  & 336    & 838145    & 1153.76 \\
  \emph{rassistischen\_Anschlag} & 78  & 15      & 0.04   & 92.96     & 1138.29 \\
  \emph{nach}                  & 681   & 335175  & 134.59 & 335721.41 & 1125.60 \\
  \emph{neun}                  & 172   & 9137    & 3.73   & 9305.27   & 985.25 \\
  \emph{Hanau}                 & 100   & 3160    & 1.31   & 3258.69   & 673.52 \\
  \emph{Anschlags}             & 60    & 409     & 0.19   & 468.71    & 580.38 \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    4: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 10\tabcolsep) * \real{0.1666}}
  >{\raggedright\arraybackslash}p{(\linewidth - 10\tabcolsep) * \real{0.1666}}
  >{\raggedright\arraybackslash}p{(\linewidth - 10\tabcolsep) * \real{0.1667}}
  >{\raggedright\arraybackslash}p{(\linewidth - 10\tabcolsep) * \real{0.1666}}
  >{\raggedright\arraybackslash}p{(\linewidth - 10\tabcolsep) * \real{0.1666}}
  >{\raggedright\arraybackslash}p{(\linewidth - 10\tabcolsep) * \real{0.1667}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
\begin{quote}
\underline{in}
\end{quote}
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
\begin{quote}
\underline{Altstadt\textbar Innenstadt \textbar{}}

\underline{Parks}
\end{quote}
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
\begin{quote}
\underline{rechtsradikalen \textbar{}}

\underline{rechtsradikale\textbar rechtsextremistischen}
\end{quote}
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
\begin{quote}
\underline{Anschlag\textbar Angriff}
\end{quote}
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
\begin{quote}
\underline{Fußball}
\end{quote}
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\underline{Dresden\textbar Chemnitz\textbar Leipzig} & 11460,33**** & 13,32*** & 185,32**** & 1122,20**** & -0,02 \\
\underline{Gladbach\textbar Hoffenheim\textbar Schalke} & 189,88**** & -28,98 & -80,60 & 0,56 & 172,23**** \\
\underline{Hanau} & 1373,56**** & 5,19* & 183,96**** & 2655,73**** & -0,23 \\
\underline{Mannheim\textbar Ulm\textbar Aachen} & 14882,43**** & 58,44**** & -1,92 & -1,64 & -11,46 \\
\underline{Potsdam\textbar Magdeburg\textbar Erfurt} & 45767,35**** & 0,81 & 8,10** & 6,00* & -34,82 \\
\end{longtable}

% \phantomsection\label{_Ref162017334}{}Tabelle : Kollokationsmatrix zu Wort-Clustern des Zeitraumes 2020 -- 2021 mit \emph{Log-Likelihood}-Werten. Dargestellt ist die gerichtete Kollokationsstärke der Zeilen-Cluster zu den Spalten-Clustern. Vermerkte Signifikanzlevel: *p \textless{} .05 (LL \textgreater= 3,84), **p \textless{} .01 (LL \textgreater= 6,63), ***p \textless{} .001 (LL \textgreater= 10,38), ****p \textless{} .0001 (LL \textgreater= 15,13).
""",
        r"""\begin{sidewaystable}
\caption{Kollokationsmatrix zu Wort-Clustern des Zeitraumes 2020 -- 2021 mit \emph{Log-Likelihood}-Werten. Dargestellt ist die gerichtete Kollokationsstärke der Zeilen-Cluster zu den Spalten-Clustern. Vermerkte Signifikanzlevel: *p \textless{} .05 (LL \textgreater= 3,84), **p \textless{} .01 (LL \textgreater= 6,63), ***p \textless{} .001 (LL \textgreater= 10,38), ****p \textless{} .0001 (LL \textgreater= 15,13).}
\label{tab:collocation_matrix}
\begin{tabularx}{\textwidth}{X X X X X X}
  \lsptoprule
  & \underline{in} & \underline{Altstadt\textbar Innenstadt\textbar Parks} & \underline{rechtsra\-dikalen\textbar rechtsradikale\textbar rechtsextremistischen} & \underline{Anschlag\textbar Angriff} & \underline{Fußball} \\
  \midrule
  \underline{Dresden\textbar Chemnitz\textbar Leipzig}    & 11460.33**** & 13.32***  & 185.32**** & 1122.20**** & -0.02 \\
  \underline{Gladbach\textbar Hoffenheim\textbar Schalke} & 189.88****   & -28.98    & -80.60     & 0.56        & 172.23**** \\
  \underline{Hanau}                                       & 1373.56****  & 5.19*     & 183.96**** & 2655.73**** & -0.23 \\
  \underline{Mannheim\textbar Ulm\textbar Aachen}         & 14882.43**** & 58.44**** & -1.92      & -1.64       & -11.46 \\
  \underline{Potsdam\textbar Magdeburg\textbar Erfurt}    & 45767.35**** & 0.81      & 8.10**     & 6.00*       & -34.82 \\
  \lspbottomrule
 \end{tabularx}
\end{sidewaystable}""",
    ),
    5: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 6\tabcolsep) * \real{0.1928}}
  >{\raggedright\arraybackslash}p{(\linewidth - 6\tabcolsep) * \real{0.3237}}
  >{\raggedright\arraybackslash}p{(\linewidth - 6\tabcolsep) * \real{0.2500}}
  >{\raggedright\arraybackslash}p{(\linewidth - 6\tabcolsep) * \real{0.2335}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Ausgangsvektor
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
1999 -- 2001
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
2011 -- 2013
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
2020 -- 2021
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\emph{Querdenker}

(99 -- 01) & \emph{Querdenker} (1.0),

\emph{Pragmatiker} (0.73),

\emph{Provokateur} (0.71),

\emph{Selbstdarsteller} (0.69),

\emph{Querkopf} (0.68) & \emph{Querdenker} (0.76),

\emph{Netzwerker} (0.73),

\emph{Freigeist} (0.7),

\emph{Vordenker} (0.69),

\emph{Pragmatiker} (0.69) & \emph{Pragmatiker} (0.66),

\emph{Netzwerker} (0.66),

\emph{Vordenker} (0.66),

\emph{Liberaler} (0.65),

\emph{Intellektueller} (0.64) \\
\emph{Querdenker}

(20 -- 21) & \emph{Autonome} (0.75),

\emph{Rechtsextreme} (0.7),

\emph{Neonazis} (0.7),

\emph{Rechtsextremen} (0.69),

\emph{Rechtsradikale} (0.68) & \emph{Rechtsextreme} (0.73),

\emph{Rechtsradikale} (0.71),

\emph{Burschenschafter} (0.7),

\emph{gewaltbereite} (0.7),

\emph{Islamfeinde} (0.68) & \emph{Querdenker} (1.0),

\emph{Coronaleugner} (0.8),

\emph{Reichsbürger} (0.79),

\emph{Corona-Leugner} (0.79),

\emph{Impfgegner} (0.78) \\
\begin{minipage}[t]{\linewidth}\raggedright
\emph{Hanau}\\
(20 -- 21)\strut
\end{minipage} & \emph{Mölln} (0.63),

\emph{Guben} (0.63),

\emph{Rostock-lichtenhagen} (0.59),

\emph{Solingen} (0.58),

\emph{Düsseldorfer\_Synagoge} (0.57) & \emph{Mölln} (0.7),

\emph{Solingen} (0.66),

\emph{Rostock-lichtenhagen} (0.65),

\emph{Newtown} (0.61),

\emph{Lichtenhagen} (0.59) & \emph{Hanau} (1.0),

\emph{Christchurch} (0.66),

\emph{Mölln} (0.66),

\emph{Hanauer} (0.64),

\emph{Anschlag} (0.64) \\
\emph{Björn\_Höcke}

(20 -- 21) & \emph{Christian\_Worch} (0.67),

\emph{Horst\_Mahler} (0.65),

\emph{Npd} (0.64),

\emph{Aktionsbüro\_Norddeutschland} (0.63),

\emph{Reps} (0.63) & \emph{Holger\_Apfel} (0.81),

\emph{Udo\_Pastörs} (0.79),

\emph{Udo\_Voigt} (0.77),

\emph{Pastörs} (0.73),

\emph{NPD-Kader} (0.72) & \emph{Björn\_Höcke} (1.0),

\emph{Höcke} (0.88),

\emph{Andreas\_Kalbitz} (0.85),

\emph{Jörg\_Meuthen} (0.8),

\emph{Thüringer\_AfD-Chef\_Björn\_Höcke} (0.79) \\
\emph{Bin\_Laden}

(02 -- 04) & \emph{Bin\_Laden} (0.94),

\emph{Osama\_Bin\_Laden} (0.9),

\emph{Ussama\_Bin\_Laden} (0.87),

\emph{bin\_Laden} (0.85),

\emph{al-Qaida} (0.8) & \emph{Bin\_Laden} (0.86),

\emph{bin\_Laden} (0.82),

\emph{al-Qaida} (0.81),

\emph{Osama\_Bin\_Laden} (0.76),

\emph{Al-Qaida-Chef} (0.75) & \emph{al-Qaida} (0.69),

\emph{Terroristen} (0.66),

\emph{Soleimani} (0.64),

\emph{iS} (0.64),

\emph{Al-Qaida} (0.64) \\
\end{longtable}

% \phantomsection\label{_Ref162017640}{}Tabelle : Nächste Nachbarn einzelner Embeddings mit Cosinus-Ähnlichkeit im zeitlichen Verlauf.
""",
        r"""\begin{sidewaystable}
\caption{Nächste Nachbarn einzelner Embeddings mit Cosinus-Ähnlichkeit im zeitlichen Verlauf.}
\label{tab:diachronic_neighbors}
 \begin{tabularx}{\textwidth}{>{\hsize=0.8\hsize}X >{\hsize=1.066\hsize}X >{\hsize=1.066\hsize}X >{\hsize=1.066\hsize}X}
  \lsptoprule
  Ausgangsvektor & 1999--2001 & 2011--2013 & 2020--2021 \\
  \midrule
  \emph{Querdenker (99--01)} & \emph{Querdenker} (1.0), \emph{Pragmatiker} (0.73), \emph{Provokateur} (0.71), \emph{Selbstdarsteller} (0.69), \emph{Querkopf} (0.68) & \emph{Querdenker} (0.76), \emph{Netzwerker} (0.73), \emph{Freigeist} (0.70), \emph{Vordenker} (0.69), \emph{Pragmatiker} (0.69) & \emph{Pragmatiker} (0.66), \emph{Netzwerker} (0.66), \emph{Vordenker} (0.66), \emph{Liberaler} (0.65), \emph{Intellektueller} (0.64) \\
  \emph{Querdenker (20--21)} & \emph{Autonome} (0.75), \emph{Rechtsextreme} (0.70), \emph{Neonazis} (0.70), \emph{Rechtsextremen} (0.69), \emph{Rechtsradikale} (0.68) & \emph{Rechtsextreme} (0.73), \emph{Rechtsradikale} (0.71), \emph{Burschenschafter} (0.70), \emph{gewaltbereite} (0.70), \emph{Islamfeinde} (0.68) & \emph{Querdenker} (1.0), \emph{Coronaleugner} (0.80), \emph{Reichsbürger} (0.79), \emph{Corona-Leugner} (0.79), \emph{Impfgegner} (0.78) \\
  \emph{Hanau (20--21)} & \emph{Mölln} (0.63), \emph{Guben} (0.63), \emph{Rostock-lichtenhagen} (0.59), \emph{Solingen} (0.58), \emph{Düsseldorfer\_Synagoge} (0.57) & \emph{Mölln} (0.70), \emph{Solingen} (0.66), \emph{Rostock-lichtenhagen} (0.65), \emph{Newtown} (0.61), \emph{Lichtenhagen} (0.59) & \emph{Hanau} (1.0), \emph{Christchurch} (0.66), \emph{Mölln} (0.66), \emph{Hanauer} (0.64), \emph{Anschlag} (0.64) \\
  \emph{Björn\_Höcke (20--21)} & \emph{Christian\_Worch} (0.67), \emph{Horst\_Mahler} (0.65), \emph{NPD} (0.64), \emph{Aktionsbüro\_Norddeutschland} (0.63), \emph{Reps} (0.63) & \emph{Holger\_Apfel} (0.81), \emph{Udo\_Pastörs} (0.79), \emph{Udo\_Voigt} (0.77), \emph{Pastörs} (0.73), \emph{NPD-Kader} (0.72) & \emph{Björn\_Höcke} (1.0), \emph{Höcke} (0.88), \emph{Andreas\_Kalbitz} (0.85), \emph{Jörg\_Meuthen} (0.80), \emph{Thüringer\_AfD-Chef\_Björn\_Höcke} (0.79) \\
  \emph{Bin\_Laden (02--04)} & \emph{Bin\_Laden} (0.94), \emph{Osama\_Bin\_Laden} (0.90), \emph{Ussama\_Bin\_Laden} (0.87), \emph{bin\_Laden} (0.85), \emph{al-Qaida} (0.80) & \emph{Bin\_Laden} (0.86), \emph{bin\_Laden} (0.82), \emph{al-Qaida} (0.81), \emph{Osama\_Bin\_Laden} (0.76), \emph{Al-Qaida-Chef} (0.75) & \emph{al-Qaida} (0.69), \emph{Terroristen} (0.66), \emph{Soleimani} (0.64), \emph{iS} (0.64), \emph{Al-Qaida} (0.64) \\
  \lspbottomrule
 \end{tabularx}
\end{sidewaystable}""",
    ),
    6: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 2\tabcolsep) * \real{0.3751}}
  >{\raggedright\arraybackslash}p{(\linewidth - 2\tabcolsep) * \real{0.6249}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Cluster
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Enthaltene Wörter
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\underline{14\_rechtsradikale} \underline{\textbar{}}

\underline{rechtsradikalen} \underline{\textbar{}}

\underline{rechtsextreme} & \emph{rechtsradikale, rechtsradikalen, rechtsextreme, rechtspopulistischen, rechtsextremen, rechte, rassistischen, fremdenfeindliche, fremdenfeindlichen, nationalistische, nationalistischen, rechtspopulistische, rechter, antisemitischen, rassistische, antisemitische, rechten, radikalen, radikale, linker, Erstarken} \\
\underline{14\_Asylbewerberunterkunft} \underline{\textbar{}}

\underline{Asylunterkunft\textbar{}}

\underline{Asylbewerberheim} & \emph{Asylbewerberunterkunft, Asylunterkunft, Asylbewerberheim, Gaststätte, Flüchtlingsheim, Lagerhalle, Mehrfamilienhaus, Scheune, Kaserne, Brandstiftung, Synagoge} \\
\underline{14\_Heidenau} \underline{\textbar{}}

\underline{Freital} \underline{\textbar{}}

\underline{Bautzen} & \emph{Heidenau, Freital, Bautzen, Clausnitz, Tröglitz, Hoyerswerda} \\
\end{longtable}

% \phantomsection\label{_Ref162017162}{}Tabelle : Drei Beispiel-Cluster des Zeitraums 2014 -- 2016 und alle enthaltenen Wörter (gereiht nach Nähe zum Cluster-Centroid).
""",
        r"""\begin{table}
\caption{Drei Beispiel-Cluster des Zeitraums 2014 -- 2016 und alle enthaltenen Wörter (gereiht nach Nähe zum Cluster-Centroid).}
\label{tab:example_clusters}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=1.5\hsize}X}
  \lsptoprule
  Cluster & Enthaltene Wörter \\
  \midrule
  \underline{rechtsradikale \textbar rechtsradikalen \textbar rechtsextreme} & \emph{rechtsradikale, rechtsradikalen, rechtsextreme, rechtspopulistischen, rechtsextremen, rechte, rassistischen, fremdenfeindliche, fremdenfeindlichen, nationalistische, nationalistischen, rechtspopulistische, rechter, antisemitischen, rassistische, antisemitische, rechten, radikalen, radikale, linker, Erstarken} \\
  
  \underline{Asylbewerber\-unterkunft \textbar Asylunterkunft \textbar Asylbewerberheim} & \emph{Asylbewerberunterkunft, Asylunterkunft, Asylbewerberheim, Gaststätte, Flüchtlingsheim, Lagerhalle, Mehrfamilienhaus, Scheune, Kaserne, Brandstiftung, Synagoge} \\
  
  \underline{Heidenau \textbar Freital \textbar Bautzen} & \emph{Heidenau, Freital, Bautzen, Clausnitz, Tröglitz, Hoyerswerda} \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    7: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 2\tabcolsep) * \real{0.3022}}
  >{\raggedright\arraybackslash}p{(\linewidth - 2\tabcolsep) * \real{0.6978}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Cosinus-Ähnlichkeit
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Cluster
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
sehr hoch (\textgreater= 0.53) & \underline{ta\_Extremisten\textbar Islamisten\textbar Dschihadisten} (0.78)

\underline{ta\_Hisbollah\textbar Hamas\textbar Pkk} (0.73)

\underline{ta\_Terrorismus\textbar Terror} (0.65)

\underline{ta\_Rechtsextremisten\textbar Neonazis\textbar Rechtsextremen}

(0.53) \\
eher hoch (\textasciitilde{} 0.27) & \underline{ta\_Rede} (0.27)

\underline{ta\_Verbreitung} (0.27)

\underline{ta\_Unterschrift} (0.27)

\underline{ta\_Rot}-grün\underline{\textbar Schwarz}-gelb (0.27) \\
eher niedrig (\textasciitilde{} 0.14) & \underline{ta\_Krankenkassen\textbar Kassen} (0.14)

\underline{ta\_Privatisierung} (0.14)

\underline{ta\_Plänen} (0.14)

\underline{ta\_Sachbearbeiterin\textbar Sekretärin\textbar Putzfrau} (0.14) \\
sehr niedrig (\textless= -0.31) & \underline{ta\_samt\textbar nebst\textbar inklusive} (-0.31)

\underline{ta\_bietet} (-0.31)

\underline{ta\_wunderbare\textbar tolle\textbar schöne} (-0.39)

\underline{ta\_schwärmt} (-0.39) \\
\end{longtable}

% \phantomsection\label{_Ref162017170}{}Tabelle : Beispiele für Cluster mit unterschiedlicher Distanz zum Extremismus-Centroid.
""",
        r"""\begin{table}
\caption{Beispiele für Cluster mit unterschiedlicher Distanz zum Extremismus-Centroid.}
\label{tab:cluster_distances_centroid}
 \begin{tabularx}{\textwidth}{>{\hsize=0.6\hsize}X >{\hsize=1.4\hsize}X}
  \lsptoprule
  Cosinus-Ähnlichkeit & Cluster \\
  \midrule
  sehr hoch (\textgreater= 0.53) & \underline{ta\_Extremisten\textbar Islamisten\textbar Dschihadisten} (0.78) \\
                                & \underline{ta\_Hisbollah\textbar Hamas\textbar Pkk} (0.73) \\
                                & \underline{ta\_Terrorismus\textbar Terror} (0.65) \\
                                & \underline{ta\_Rechtsextremisten\textbar Neonazis\textbar Rechtsextremen} (0.53) \\
  \midrule
  eher hoch (\textasciitilde{} 0.27) & \underline{ta\_Rede} (0.27) \\
                                     & \underline{ta\_Verbreitung} (0.27) \\
                                     & \underline{ta\_Unterschrift} (0.27) \\
                                     & \underline{ta\_Rot-grün\textbar Schwarz-gelb} (0.27) \\
  \midrule
  eher niedrig (\textasciitilde{} 0.14) & \underline{ta\_Krankenkassen\textbar Kassen} (0.14) \\
                                        & \underline{ta\_Privatisierung} (0.14) \\
                                        & \underline{ta\_Plänen} (0.14) \\
                                        & \underline{ta\_Sachbearbeiterin\textbar Sekretärin\textbar Putzfrau} (0.14) \\
  \midrule
  sehr niedrig (\textless= -0.31) & \underline{ta\_samt\textbar nebst\textbar inklusive} (-0.31) \\
                                  & \underline{ta\_bietet} (-0.31) \\
                                  & \underline{ta\_wunderbare\textbar tolle\textbar schöne} (-0.39) \\
                                  & \underline{ta\_schwärmt} (-0.39) \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    8: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 8\tabcolsep) * \real{0.0770}}
  >{\raggedright\arraybackslash}p{(\linewidth - 8\tabcolsep) * \real{0.2871}}
  >{\raggedright\arraybackslash}p{(\linewidth - 8\tabcolsep) * \real{0.1363}}
  >{\raggedright\arraybackslash}p{(\linewidth - 8\tabcolsep) * \real{0.3537}}
  >{\raggedright\arraybackslash}p{(\linewidth - 8\tabcolsep) * \real{0.1460}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Rang
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Syntagmatisch benachbarte Cluster
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
\emph{Log-Likelihood}
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Paradigmatisch benachbarte Cluster
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Cosinus-Ähnlichkeit
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
1 & \underline{11\_wurden} & 9420.50 & \underline{11\_festgenommen\_worden} & 0.82 \\
2 & \underline{11\_wurde} & 8247.68 & \underline{11\_getötet\textbar{}}

\underline{erschossen} & 0.76 \\
3 & \underline{11\_Polizei} & 4277.99 & \underline{11\_inhaftiert\textbar{}}

\underline{hingerichtet\textbar{}}

\underline{gefoltert} & 0.72 \\
4 & \underline{11\_Verdächtigen\textbar{}}

\underline{Tatverdächtigen\textbar{}}

\underline{mutmaßlichen\_Täter} & 1983.41 & \underline{11\_beschuldigt\textbar{}}

\underline{verdächtigt\textbar{}}

\underline{angeklagt} & 0.70 \\
5 & \underline{11\_Personen} & 1749.90 & \underline{11\_geschnappt\textbar{}}

\underline{verhört\textbar{}}

\underline{vernommen} & 0.69 \\
6 & \underline{11\_sieben\textbar sechs\textbar fünf} & 1741.97 & \underline{11\_ermordet\textbar{}}

\underline{umgebracht\textbar{}}

\underline{entführt} & 0.69 \\
7 & \underline{11\_Hund\textbar Mann\textbar Tier} & 1333.86 & \underline{11\_Entführung\textbar{}}

\underline{Tötung\textbar{}}

\underline{Ermordung} & 0.61 \\
8 & \underline{11\_31}-Jähriger\underline{\textbar{}}

\underline{27}-Jähriger\underline{\textbar{}}

\underline{44}-Jähriger & 1130.97 & \underline{11\_Durchsuchungen\textbar{}}

\underline{Razzia\textbar Razzien} & 0.59 \\
9 & \underline{11\_Demonstranten} & 1034.54 & \underline{11\_angegriffen\textbar{}}

\underline{attackiert} & 0.59 \\
10 & \underline{11\_festgenommen} & 1017.76 & \underline{11\_Verdächtigen\textbar{}}

\underline{Tatverdächtigen\textbar{}}

\underline{mutmaßlichen\_Täter} & 0.58 \\
\end{longtable}

% \phantomsection\label{_Ref162017239}{}Tabelle : Kollokationen und \emph{next neighbours} im Word-Embedding-Modell zum Cluster \underline{11\_festgenommen.}
""",
        r"""\begin{table}
\caption{Kollokationen und \emph{next neighbours} im Word-Embedding-Modell zum Cluster \underline{11\_festgenommen}.}
\label{tab:collocations_neighbors}
 \begin{tabularx}{\textwidth}{>{\hsize=0.3\hsize}X >{\hsize=1.65\hsize}X >{\hsize=0.7\hsize}X>{\hsize=1.65\hsize}X >{\hsize=0.7\hsize}X }
  \lsptoprule
  Rang & Syntagm. benachbarte Cluster & Log-Likelihood & Paradigm. benachbarte Cluster & Cosinus-Ähnlichkeit \\
  \midrule
  1  & \underline{11\_wurden}                              & 9420.50   & \underline{11\_festgenommen\_worden}                     & 0.82 \\
  2  & \underline{11\_wurde}                               & 8247.68   & \underline{11\_getötet\textbar erschossen}               & 0.76 \\
  3  & \underline{11\_Polizei}                             & 4277.99   & \underline{11\_inhaftiert\textbar hingerichtet\textbar gefoltert} & 0.72 \\
  4  & \underline{11\_Verdächtigen\textbar Tatverdächtigen\textbar mutmaßlichen\_Täter} & 1983.41 & \underline{11\_beschuldigt\textbar verdächtigt\textbar angeklagt} & 0.70 \\
  5  & \underline{11\_Personen}                            & 1749.90   & \underline{11\_geschnappt\textbar verhört\textbar vernommen} & 0.69 \\
  6  & \underline{11\_sieben\textbar sechs\textbar fünf}   & 1741.97   & \underline{11\_ermordet\textbar umgebracht\textbar entführt} & 0.69 \\
  7  & \underline{11\_Hund\textbar Mann\textbar Tier}      & 1333.86   & \underline{11\_Entführung\textbar Tötung\textbar Ermordung} & 0.61 \\
  8  & \underline{11\_31-Jähriger\textbar 27-Jähriger\textbar 44-Jähriger} & 1130.97 & \underline{11\_Durchsuchungen\textbar Razzia\textbar Razzien} & 0.59 \\
  9  & \underline{11\_Demonstranten}                      & 1034.54   & \underline{11\_angegriffen\textbar attackiert}            & 0.59 \\
  10 & \underline{11\_festgenommen}                       & 1017.76   & \underline{11\_Verdächtigen\textbar Tatverdächtigen\textbar mutmaßlichen\_Täter} & 0.58 \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    9: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1717}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1565}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.6718}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Extremismus-Variante
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Bezug
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Frame-Elemente
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\multirow{3}{=}{Rechtsextremismus} & Politik & \underline{ta\_Afd\textbar Npd}

\underline{ta\_Dvu}

\underline{ta\_Schill\textbar Schill-Partei}

\underline{ta\_Gauland\textbar Höcke\textbar Meuthen (AfD-Politiker / Möllemann)}

\underline{ta\_Alexander\_Gauland\textbar Jörg\_Meuthen\textbar Poggenburg (rechtsaußen Politiker)}

\underline{-\/-}

\underline{sp\_Afd\textbar Npd}

\underline{sp\_Gauland\textbar Petry\textbar Höcke}

\underline{-\/-}

\underline{/} \\
& NSU & \underline{ta\_Nsu}

\underline{ta\_Zschäpe\textbar Beate\_Zschäpe}

\underline{-\/-}

\underline{sp\_Beate\_Zschäpe\textbar{} Zschäpe\textbar Nsu-prozess}

\underline{sp\_Uwe\_Böhnhardt\textbar Böhnhardt\textbar Uwe\_Mundlos}

\underline{-\/-}

\underline{we\_Uwe\_Böhnhardt\textbar Uwe\_Mundlos\textbar Böhnhardt}

\underline{we\_Beate\_Zschäpe\textbar Zschäpe\textbar Nsu} \\
& außerparlamentarisch & \underline{ta\_Rechtsextremisten\textbar Neonazis\textbar Rechtsextremen}

\underline{-\/-}

\underline{sp\_Neonazis\textbar Rechtsextremisten\textbar Rechtsextreme}

\underline{-\/-}

\underline{we\_Rechtsextremisten\textbar Neonazis\textbar Rechtsextremen} \\
\multirow{2}{=}{Linksextremismus} & Politik & \underline{ta\_Linke}

\underline{ta\_Pds\textbar Linkspartei}

\underline{ta\_Oskar\_Lafontaine\textbar Lafontaine\textbar Gregor\_Gysi}

\underline{-\/-}

\underline{sp\_Linkspartei\textbar Pds\textbar Linken}

\underline{sp\_Lafontaine\textbar Gysi\textbar Oskar\_Lafontaine}

\underline{-\/-}

\underline{/} \\
& außerparlamentarisch & \underline{ta\_gegen\_Rechts\textbar antifaschistische\textbar Antifa}

\underline{ta\_Antifa-gruppen\textbar Aktivistinnen\textbar linke\_Gruppen}

\underline{ta\_Heiligendamm\textbar Genua\textbar G-8-Gipfel}

\underline{-\/-}

\underline{sp\_G-8-Gipfel\textbar G7-Gipfel\textbar G-20-Gipfel}

\underline{-\/-}

\underline{we\_G-20-Gipfel\textbar G-8-Gipfel\textbar G-7-gipfel} \\
Islamismus & Islamischer Staat & \underline{ta\_iS}

\underline{-\/-}

\underline{sp\_Miliz\textbar Milizen\textbar Armee}

\underline{-\/-}

\underline{we\_iS} \\
\end{longtable}

% \phantomsection\label{_Ref162017380}{}Tabelle : Zu betrachtende Cluster für den medialen Vergleich nach Extremismsuvariante und thematischem Bezug.
""",
        r"""\begin{table}
\caption{Zu betrachtende Cluster für den medialen Vergleich nach Extremismusvariante und thematischem Bezug}
\label{tab:cluster_newspapers}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{3}{=}{Rechts\-ex\-tre\-mis\-mus} 
  & Politik & \underline{ta\_Afd \textbar Npd, ta\_Dvu, ta\_Schill \textbar Schill-Partei, ta\_Gauland \textbar Höcke \textbar Meuthen (AfD-Politiker / Möllemann), ta\_Alexander\_Gauland \textbar Jörg\_Meuthen \textbar Poggenburg (rechtsaußen Politiker), sp\_Afd \textbar Npd, sp\_Gauland \textbar Petry \textbar Höcke} \\
  & NSU & \underline{ta\_Nsu, ta\_Zschäpe \textbar Beate\_Zschäpe, sp\_Beate\_Zschäpe \textbar Zschäpe \textbar Nsu-prozess, sp\_Uwe\_Böhnhardt \textbar Böhnhardt \textbar Uwe\_Mundlos, we\_Uwe\_Böhnhardt \textbar Uwe\_Mundlos \textbar Böhnhardt, we\_Beate\_Zschäpe \textbar Zschäpe \textbar Nsu} \\
  & außerparlamentarisch & \underline{ta\_Rechtsextremisten \textbar Neonazis \textbar Rechtsextremen, sp\_Neonazis \textbar Rechtsextremisten \textbar Rechtsextreme, we\_Rechtsextremisten \textbar Neonazis \textbar Rechtsextremen} \\
  \midrule
  \multirow{2}{=}{Links\-ex\-tre\-mis\-mus} 
  & Politik & \underline{ta\_Linke, ta\_Pds \textbar Linkspartei, ta\_Oskar\_Lafontaine \textbar Lafontaine \textbar Gregor\_Gysi, sp\_Linkspartei \textbar Pds \textbar Linken, sp\_Lafontaine \textbar Gysi \textbar Oskar\_Lafontaine} \\
  & außerparlamentarisch & \underline{ta\_gegen\_Rechts \textbar antifaschistische \textbar Antifa, ta\_Antifa-gruppen \textbar Aktivistinnen \textbar linke\_Gruppen, ta\_Heiligendamm \textbar Genua \textbar G-8-Gipfel, sp\_G-8-Gipfel \textbar G7-Gipfel \textbar G-20-Gipfel, we\_G-20-Gipfel \textbar G-8-Gipfel \textbar G-7-Gipfel} \\
  \midrule
  Islamismus & Islamischer Staat & \underline{ta\_iS, sp\_Miliz \textbar Milizen \textbar Armee, we\_iS} \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    10: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.2933}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1359}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.5708}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Extremismus-Variante
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Bezug
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Frame-Elemente
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\multirow{2}{=}{Rechtsextremismus} & historisch & \underline{Holocaust\textbar Nazis\textbar Nationalsozialismus}

\underline{Ss\textbar Nazi-deutschland\textbar Nazi-zeit (Zeit des Nationalsozialismus \& weitere Kriege)}

\underline{1945\textbar Zweiten\_Weltkrieg} \\
& zeitgenössisch & \underline{Npd}

\underline{Antisemitismus}

\underline{Intoleranz\textbar Hetze\textbar Rassismus (Merkmale des {[}Rechts-{]}Extremismus \& Diskriminierung)} \\
Linksextremismus & zeitgenössisch & \underline{Pds} \\
\multirow{2}{=}{Politischer Extremismus} & historisch & \underline{Weimarer\_Republik\textbar Nationalsozialisten\textbar Sowjetunion (Totalitarismus / 20. Jahrhundert)}

\underline{Liberalismus\textbar Nationalismus\textbar Totalitarismus} \\
& zeitgenössisch & \underline{Autonomen\textbar Antifa\textbar Rechtsextremisten}

\underline{Schill-Partei\textbar Schill\textbar GAL} \\
\multirow{2}{=}{Islamismus} & Nahostkonflikt & \underline{Tulkarm\textbar Tulkarem\textbar jüdische\_Siedler}

\underline{Hebron\textbar Nablus\textbar Rafah}

\underline{Palästinenserpräsident\_Jassir\_Arafat\textbar Jassir\_Arafat\textbar Abbas}

\underline{Gaza\textbar Ramallah\textbar Tel\_Aviv}

\underline{Gaza-streifen\textbar Westjordanland\textbar Gazastreifen}

\underline{Israelis\textbar Palästinenser\textbar Palästinensern}

\underline{israelischen\textbar israelische\textbar palästinensische} \\
& Irakkrieg & \underline{Al\_Qaida\textbar al-Qaida\textbar Extremisten}

\underline{Qaida\textbar Terrorgruppe\textbar Terrororganisation}

\underline{Taliban-Kämpfer\textbar Al-Qaida-Kämpfer\textbar Afghanen}

\underline{Saddam\_Hussein\textbar Saddam} \\
\end{longtable}

% \phantomsection\label{_Ref162017395}{}Tabelle : Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2002 -- 2004.
""",
        r"""\begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2002--2004}
\label{tab:2002}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{2}{=}{Rechts\-ex\-tre\-mis\-mus} 
  & historisch & \underline{Holocaust \textbar Nazis \textbar Nationalsozialismus, Ss \textbar Nazi-deutschland \textbar Nazi-zeit, 1945 \textbar Zweiten\_Weltkrieg} \\
  & zeitgenössisch & \underline{Npd, Antisemitismus, Intoleranz \textbar Hetze \textbar Rassismus} \\
  \midrule
  Linksextremismus & zeitgenössisch & \underline{Pds} \\
  \midrule
  \multirow{2}{=}{Politischer Extremismus} 
  & historisch & \underline{Weimarer\_Republik \textbar Nationalsozialisten \textbar Sowjetunion, Liberalismus \textbar Nationalismus \textbar Totalitarismus} \\
  & zeitgenössisch & \underline{Autonomen \textbar Antifa \textbar Rechtsextremisten, Schill-Partei \textbar Schill \textbar GAL} \\
  \midrule
  \multirow{2}{=}{Islamismus} 
  & Nahostkonflikt & \underline{Tulkarm \textbar Tulkarem \textbar jüdische\_Siedler, Hebron \textbar Nablus \textbar Rafah, Palästinenserpräsident\_Jassir\_Arafat \textbar Jassir\_Arafat \textbar Abbas, Gaza \textbar Ramallah \textbar Tel\_Aviv, Gaza-streifen \textbar Westjordanland \textbar Gazastreifen, Israelis \textbar Palästinenser \textbar Palästinensern, israelischen \textbar israelische \textbar palästinensische} \\
  & Irakkrieg & \underline{Al\_Qaida \textbar al-Qaida \textbar Extremisten, Qaida \textbar Terrorgruppe \textbar Terrororganisation, Taliban-Kämpfer \textbar Al-Qaida-Kämpfer \textbar Afghanen, Saddam\_Hussein \textbar Saddam} \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    11: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.2337}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1825}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.5838}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Extremismus-Variante
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Bezug
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Frame-Elemente
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\multirow{2}{=}{Rechtsextremismus} & historisch & \underline{Nationalsozialismus\textbar Ns-zeit\textbar Holocaust,}

\underline{Nazis\textbar Hitler,}

\underline{1945\textbar Zweiten\_Weltkrieg\textbar Kriegsende (Kriegsende zweiter Weltkrieg)} \\
& zeitgenössisch & \underline{Rechtsextremen\textbar Rechtsextremisten\textbar Neonazis (rechtsextreme Akteure)} \\
\multirow{2}{=}{Linksextremismus} & zeitgenössisch & \underline{Pds\textbar Linkspartei\textbar WASG (linksaußen Parteien),}

\underline{Heiligendamm\textbar G-8-Gipfel} \\
& historisch & \underline{Raf} \\
\multirow{2}{=}{Politischer Extremismus} & historisch & \underline{Kommunismus\textbar Sowjetunion\textbar Weimarer\_Republik (Totalitarismus, 68er, Weimarer Republik)}

\underline{KZs\textbar Konzentrationslager\textbar Deportationen (NS- und DDR-Zeit),}

\underline{Ss\textbar Gestapo\textbar Wehrmacht (staatliche Akteure NS- \& DDR-Zeit)} \\
& zeitgenössisch & \underline{Wahlalternative\textbar LinksparteiPDS\textbar Wahlalternative\_Arbeit (rechte und linke Parteien und Organisationen),}

\underline{Gegendemonstranten\textbar Antifas\textbar Kundgebungen (Politischer Extremismus),} \\
\multirow{2}{=}{Islamismus} & Nahostkonflikt & \underline{Hisbollah-miliz\textbar Luftangriffe\textbar israelische\_Armee (kriegerische Handlungen \& Nahostkonflikt)}

\underline{Mahmud\_Abbas\textbar Palästinenserpräsident\_Abbas\textbar Präsident\_Mahmud\_Abbas (Nahostkonflikt)}

\underline{Gazastreifen\textbar Gaza-streifen\textbar Westjordanland (palästinensische Gebiete / Südlibanon)}

\underline{Palästinenserpräsident\_Mahmud\_Abbas\textbar Palästinensern\textbar Ramallah}

\underline{Beirut\textbar Gaza\textbar Tel\_Aviv (Orte des Nahostkonfliktes / Libanonkrieges)}

\underline{Olmert\textbar Abbas\textbar Scharon (Präsidenten im Nahostkonflikt)}

\underline{Hamas\textbar Hisbollah\textbar Fatah (islamistische Gruppen / Fattah)}

\underline{Palästinenser\textbar Israelis} \\
& Irakkrieg & \underline{Qaida\textbar Todesschwadronen\textbar Terrorgruppen (Akteure Irakkrieg)}

\underline{Saddam\_Hussein\textbar Saddam} \\
variantenübergreifend & allgemein & \underline{rechtsextremen\textbar rechtsextremistischen\textbar rechtsextreme (Extremismus),}

\underline{Intoleranz\textbar Antisemitismus\textbar Hetze (Charakteristika von Extremismus, Unterdrückung)} \\
\end{longtable}

% \phantomsection\label{_Ref162017417}{}Tabelle : Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2005 -- 2007.
""",
        r"""\begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2005--2007}
\label{tab:2005}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{2}{=}{Rechts\-ex\-tre\-mis\-mus} 
  & historisch & \underline{Nationalsozialismus \textbar Ns-zeit \textbar Holocaust, Nazis \textbar Hitler, 1945 \textbar Zweiten\_Weltkrieg \textbar Kriegsende} \\
  & zeitgenössisch & \underline{Rechtsextremen \textbar Rechtsextremisten \textbar Neonazis} \\
  \midrule
  \multirow{2}{=}{Links\-ex\-tre\-mis\-mus} 
  & zeit\-ge\-nössisch & \underline{Pds \textbar Linkspartei \textbar WASG, Heiligendamm \textbar G-8-Gipfel} \\
  & historisch & \underline{Raf} \\
  \midrule
  \multirow{2}{=}{Politischer Extremismus} 
  & historisch & \underline{Kommunismus \textbar Sowjetunion \textbar Weimarer\_Republik, KZs \textbar Konzentrationslager \textbar Deportationen, Ss \textbar Gestapo \textbar Wehrmacht} \\
  & zeitgenössisch & \underline{Wahlalternative \textbar LinksparteiPDS \textbar Wahlalternative\_Arbeit, Gegendemonstranten \textbar Antifas \textbar Kundgebungen} \\
  \midrule
  \multirow{2}{=}{Islamismus} 
  & Nahostkonflikt & \underline{Hisbollah-miliz \textbar Luftangriffe \textbar israelische\_Armee, Mahmud\_Abbas \textbar Palästinenserpräsident\_Abbas, Gazastreifen \textbar Gaza-streifen \textbar Westjordanland, Palästinenserpräsident\_Mahmud\_Abbas \textbar Ramallah, Beirut \textbar Gaza \textbar Tel\_Aviv, Olmert \textbar Abbas \textbar Scharon, Hamas \textbar Hisbollah \textbar Fatah, Palästinenser \textbar Israelis} \\
  & Irakkrieg & \underline{Qaida \textbar Todesschwadronen \textbar Terrorgruppen, Saddam\_Hussein \textbar Saddam} \\
  \midrule
  variantenübergreifend & allgemein & \underline{rechtsextremen \textbar rechtsextremistischen \textbar rechtsextreme, Intoleranz \textbar Antisemitismus \textbar Hetze} \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    12: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1717}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1408}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.6875}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Extremismus-Variante
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Bezug
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Frame-Elemente
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\multirow{2}{=}{Rechtsextremismus} & historisch & \underline{Holocaust\textbar Nationalsozialismus} \\
& & \underline{Rechtsextremisten\textbar Neonazis\textbar Rechtsextremen (Rechtsextreme Akteure)}

\underline{Npd}

\underline{Rassismus\textbar Antisemitismus}

\underline{Sarrazin\textbar Clement (durch Parteiausschlussverfahren bedrohte SPD-Politiker)} \\
\multirow{2}{=}{Linksextremismus} & zeitgenössisch & \underline{Linkspartei\textbar Linken}

\underline{Linke} \\
& historisch & \underline{Stasi}

\underline{kommunistischen\textbar sozialistischen\textbar Sowjetunion} \\
\multirow{2}{=}{Politischer Extremismus} & historisch & \underline{des\_Zweiten\_Weltkriegs\textbar Krieges\textbar des\_Kalten\_Krieges (zweiter Weltkrieg \& Kalter Krieg)}

\underline{Hitler\textbar Adolf\_Hitler\textbar Stalin}

\underline{Nazizeit\textbar des\_Dritten\_Reiches\textbar Nazi-deutschland (NS-Zeit \& 20. Jhdt.)}

\underline{Nationalsozialisten\textbar Ss\textbar Nazis (NS-Zeit \& RAF)} \\
& zeitgenössisch & \underline{Autonomen\textbar Antifas\textbar Antifa (Akteure politischer Extremismus)}

\underline{Pds\textbar WASG\textbar Dkp} \\
\multirow{2}{=}{Islamismus} & Nahostkonflikt & \underline{Raketenbeschuss\textbar israelischen\_Armee\textbar Raketenangriffe (Nahostkonflikt)}

\underline{Gaza-streifen\textbar Gazastreifen\textbar Westjordanland}

\underline{Hamas\textbar Hisbollah\textbar Pkk (terroristische Organisationen)}

\underline{Abbas\textbar Netanjahu\textbar Palästinenserpräsident\_Mahmud\_Abbas (politische Akteure Nahost)}

\underline{Palästinenser\textbar Israelis} \\
& Irakkrieg & \underline{Qaida\textbar Osama\_Bin\_Laden\textbar afghanischen\_Taliban}

\underline{Extremisten\textbar Taliban\textbar Islamisten} \\
\end{longtable}

% \phantomsection\label{_Toc162477804}{}Tabelle : Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2008 -- 2010.
""",
        r"""\begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2008--2010}
\label{tab:2008}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{2}{=}{Rechts\-ex\-tre\-mis\-mus} 
  & historisch & \underline{Holocaust \textbar Nationalsozialismus} \\
  & zeitgenössisch & \underline{Rechtsextremisten \textbar Neonazis \textbar Rechtsextremen, Npd, Rassismus \textbar Antisemitismus, Sarrazin \textbar Clement} \\
  \midrule
  \multirow{2}{=}{Links\-ex\-tre\-mis\-mus} 
  & zeit\-ge\-nössisch & \underline{Linkspartei \textbar Linken, Linke} \\
  & historisch & \underline{Stasi, kommunistischen \textbar sozialistischen \textbar Sowjetunion} \\
  \midrule
  \multirow{2}{=}{Politischer Extremismus} 
  & historisch & \underline{des\_Zweiten\_Weltkriegs \textbar Krieges \textbar des\_Kalten\_Krieges, Hitler \textbar Adolf\_Hitler \textbar Stalin, Nazizeit \textbar des\_Dritten\_Reiches \textbar Nazi-deutschland, Nationalsozialisten \textbar Ss \textbar Nazis} \\
  & zeitgenössisch & \underline{Autonomen \textbar Antifas \textbar Antifa, Pds \textbar WASG \textbar Dkp} \\
  \midrule
  \multirow{2}{=}{Islamismus} 
  & Nahostkonflikt & \underline{Raketenbeschuss \textbar israelischen\_Armee \textbar Raketenangriffe, Gaza-streifen \textbar Gazastreifen \textbar Westjordanland, Hamas \textbar Hisbollah \textbar Pkk, Abbas \textbar Netanjahu \textbar Palästinenserpräsident\_Mahmud\_Abbas, Palästinenser \textbar Israelis} \\
  & Irakkrieg & \underline{Qaida \textbar Osama\_Bin\_Laden \textbar afghanischen\_Taliban, Extremisten \textbar Taliban \textbar Islamisten} \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    13: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1717}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1408}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.6875}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Extremismus-Variante
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Bezug
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Frame-Elemente
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\multirow{2}{=}{Rechtsextremismus} & historisch & \underline{Holocaust\textbar Nationalsozialismus}

\underline{Konzentrationslager\textbar Auschwitz\textbar Ss (NS-Zeit)}

\underline{Nazis}

\underline{1939\textbar1940\textbar1941 (Jahre der NS-Zeit \& Sowjetunion)}

\underline{Zweiten\_Weltkrieg} \\
& zeitgenössisch & \underline{Neonazis\textbar Rechtsextremisten\textbar Rechtsextremen (Akteure Rechtsextremismus \& Gegendemonstranten)}

\underline{Hells\_Angels\textbar Bandidos\textbar Rocker}

\underline{Terrorzelle\textbar Zwickauer\_Zelle\textbar Zwickauer\_Terrorzelle (NSU-Komplex)}

\underline{Npd}

\underline{Grass\textbar Thilo\_Sarrazin\textbar Sarrazin} \\
\multirow{2}{=}{Linksextremismus} & zeitgenössisch & \underline{Linkspartei\textbar Linken\textbar Linke}

\underline{Sahra\_Wagenknecht\textbar Wagenknecht\textbar Dietmar\_Bartsch (Politiker*innen Linkspartei)} \\
& historisch & \underline{Ddr} \\
\multirow{2}{=}{Politischer Extremismus} & historisch & \underline{Nsdap\textbar Sed\textbar Raf (NSDAP / SED / RAF / Weimarer Republik)}

\underline{Hitler\textbar Adolf\_Hitler\textbar Stalin (Akteure Totalitarismus \& Jude)}

\underline{Faschismus\textbar Nationalismus\textbar Antisemitismus (Faschismus / Sozialismus)} \\
& zeitgenössisch & \underline{rechtsextreme\textbar rechtsradikalen\textbar Rechtsradikalen (Attribute pol. Extrmeismus, vereinzelt Islamismus)} \\
\multirow{3}{=}{Islamismus} & Nahostkonflikt & \underline{Ostjerusalem\textbar Westbank\textbar jüdischen\_Siedlungen (Nahostkonflikt)}

\underline{Gazastreifen\textbar Gaza-streifen\textbar Westjordanland (Orte Nahostkonflikt)}

\underline{Abbas\textbar Netanjahu\textbar Hamas (politische Akteure Nahostkonflikt)}

\underline{Palästinenser\textbar Israelis\textbar Israel}

\underline{Beirut\textbar Tel\_Aviv\textbar Jerusalem (Haupt-)Städte Nahost} \\
& Syrien & \underline{syrische\_Opposition\textbar syrische\_Regierung\textbar Präsident\_Assad (pol. Akteure Syrienkrieg)}

\underline{Assads\textbar Präsident\_Baschar\_al-Assad\textbar Regimes (Regime Assad \& Gaddafi)}

\underline{Assad-Truppen\textbar Assads\_Truppen\textbar syrischen\_Armee (militärische Auseinandersetzungen Syrienkrieg)} \\
& allgemein & \underline{Extremisten\textbar Islamisten\textbar Terroristen (Akteure Islamismus / Extremismus)}

\underline{Al-kaida\textbar Al-Qaida\textbar Terrororganisation (Akteure Islamismus)}

\underline{Taliban} \\
\end{longtable}

% \phantomsection\label{_Toc162477805}{}Tabelle : Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2011 -- 2013.
""",
        r"""\begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2011--2013}
\label{tab:2011}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{2}{=}{Rechts\-ex\-tre\-mis\-mus} 
  & historisch & \underline{Holocaust \textbar Nationalsozialismus, Konzentrationslager \textbar Auschwitz \textbar Ss, Nazis, 1939 \textbar 1940 \textbar 1941, Zweiten\_Weltkrieg} \\
  & zeitgenössisch & \underline{Neonazis \textbar Rechtsextremisten \textbar Rechtsextremen, Hells\_Angels \textbar Bandidos \textbar Rocker, Terrorzelle \textbar Zwickauer\_Zelle \textbar Zwickauer\_Terrorzelle, Npd, Grass \textbar Thilo\_Sarrazin \textbar Sarrazin} \\
  \midrule
  \multirow{2}{=}{Links\-ex\-tre\-mis\-mus} 
  & zeit\-ge\-nössisch & \underline{Linkspartei \textbar Linken \textbar Linke, Sahra\_Wagenknecht \textbar Wagenknecht \textbar Dietmar\_Bartsch} \\
  & historisch & \underline{Ddr} \\
  \midrule
  \multirow{2}{=}{Politischer Extremismus} 
  & historisch & \underline{Nsdap \textbar Sed \textbar Raf, Hitler \textbar Adolf\_Hitler \textbar Stalin, Faschismus \textbar Nationalismus \textbar Antisemitismus} \\
  & zeitgenössisch & \underline{rechtsextreme \textbar rechtsradikalen \textbar Rechtsradikalen} \\
  \midrule
  \multirow{3}{=}{Islamismus} 
  & Nahostkonflikt & \underline{Ostjerusalem \textbar Westbank \textbar jüdischen\_Siedlungen, Gazastreifen \textbar Gaza-streifen \textbar Westjordanland, Abbas \textbar Netanjahu \textbar Hamas, Palästinenser \textbar Israelis \textbar Israel, Beirut \textbar Tel\_Aviv \textbar Jerusalem} \\
  & Syrien & \underline{syrische\_Opposition \textbar syrische\_Regierung \textbar Präsident\_Assad, Assads \textbar Präsident\_Baschar\_al-Assad \textbar Regimes, Assad-Truppen \textbar Assads\_Truppen \textbar syrischen\_Armee} \\
  & allgemein & \underline{Extremisten \textbar Islamisten \textbar Terroristen, Al-kaida \textbar Al-Qaida \textbar Terrororganisation, Taliban} \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    14: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1717}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1408}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.6875}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Extremismus-Variante
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Bezug
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Frame-Elemente
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\multirow{2}{=}{Rechtsextremismus} & historisch & \underline{Holocaust\textbar Nationalsozialismus}

\underline{Konzentrationslager\textbar Konzentrationslagern\textbar Auschwitz-birkenau}

\underline{Wehrmacht\textbar Stalin\textbar Roten\_Armee (Akteure NS-Zeit)}

\underline{Auschwitz}

\underline{Nazis} \\
& zeitgenössisch & \underline{Neonazis\textbar Rechtsextremisten\textbar Rechtsextreme (Akteure Rechtsextremismus, Salafisten, Gegendemonstranten)}

\underline{Npd}

\underline{Pegida}

\underline{Uwe\_Böhnhardt\textbar Uwe\_Mundlos\textbar Böhnhardt (Mundlos, Böhnhardt, NSU)}

\underline{Beate\_Zschäpe\textbar Zschäpe\textbar Nsu-prozess}

\underline{Afd}

\underline{Frauke\_Petry\textbar Petry\textbar Gauland (Politiker AfD)}

\underline{Front\_national\textbar Marine\_Le\_Pen\textbar FN}

\underline{Rassismus\textbar Homophobie\textbar Sexismus (Charakteristika Rechtsextremismus)}

\underline{rechtsradikale\textbar rechtsradikalen\textbar rechtsextreme (attributive Charakteristika Rechtsextremismus)}

\underline{Heidenau\textbar Freital\textbar Bautzen (Orte rechter Anschläge in Ostdeutschland)}

\underline{Ferguson\textbar Michael\_Brown\textbar Polizeigewalt} \\
\multirow{2}{=}{Linksextremismus} & zeitgenössisch & \underline{Linke} \\
& historisch & \underline{Ddr} \\
\multirow{2}{=}{Politischer Extremismus} & historisch & \underline{Udssr\textbar Oktoberrevolution\textbar Stalins (20. Jhdt)}

\underline{Faschisten\textbar Nationalisten\textbar Kommunisten (Anhänger Totalitarismus)}

\underline{Weimarer\_Republik\textbar Brd\textbar Ostberlin (Deutschland 20. Jhdt)}

\underline{Kommunismus\textbar Sozialismus\textbar Diktatur (Totalitarismus)} \\
& zeitgenössisch & \underline{Antifa\textbar Pegida-Bewegung\textbar Islam- (PEGIDA-Demonstrationen und -Gegendemonstrationen)} \\
\multirow{3}{=}{Islamismus} & Nahostkonflikt & \underline{Ostjerusalem\textbar Israels\_Armee\textbar radikalislamischen (Nahostkonflikt)}

\underline{Hamas}

\underline{Abbas\textbar Netanjahu\textbar Palästinensern}

\underline{Gazastreifen\textbar Westjordanland}

\underline{Jerusalem\textbar Tel\_Aviv\textbar Gaza}

\underline{israelischen\textbar palästinensische\textbar israelische}

\underline{Palästinenser\textbar Israelis} \\
& Syrienkrieg / Islamischer Staat & \underline{Baschar\_al-Assad\textbar Machthaber\_Baschar\_al-Assad\textbar Assads}

\underline{syrischen\_Rebellen\textbar syrische\_Rebellen\textbar IS- Dschihadisten (Syrienkrieg)}

\underline{Assad}

\underline{Regime\textbar Assad-regime\textbar militärisch}

\underline{Al-Nusra-Front\textbar Nusra-Front\textbar al-Qaida (Terrororganisationen Syrien)}

\underline{IS-Miliz\textbar Terrormiliz\textbar Isis (IS \& PKK)}

\underline{iS}

\underline{islamischen\_Staates\textbar islamischen\_Staats\textbar islamischen\_Staates\_iS}

\underline{kurdische\_Kämpfer\textbar irakischen\_Armee\textbar Peschmerga}

\underline{Kämpfer\textbar Is-kämpfer}

\underline{Kobane\textbar Mossul\textbar Kobani} \\
& allgemein & \underline{Extremisten\textbar Dschihadisten\textbar Islamisten} \\
variantenübergreifend & Diverses & \underline{mutmaßlichen\_Terroristen\textbar Abdelhamid\_Abaaoud\textbar Abaaoud} \\
\end{longtable}

% \phantomsection\label{_Toc162477806}{}Tabelle : Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2014 -- 2016.
""",
        r"""\begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2014--2016}
\label{tab:2014_1}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{2}{=}{Rechts\-ex\-tre\-mis\-mus} 
  & historisch & \underline{Holocaust \textbar Nationalsozialismus, Konzentrationslager \textbar Auschwitz-birkenau \textbar Ss, Wehrmacht \textbar Stalin \textbar Roten\_Armee, Auschwitz, Nazis} \\
  & zeitgenössisch & \underline{Neonazis \textbar Rechtsextremisten \textbar Rechtsextreme, Npd, Pegida, Uwe\_Böhnhardt \textbar Uwe\_Mundlos \textbar Böhnhardt, Beate\_Zschäpe \textbar Nsu-prozess, Afd, Frauke\_Petry \textbar Petry \textbar Gauland, Front\_national \textbar Marine\_Le\_Pen \textbar FN, Rassismus \textbar Homophobie \textbar Sexismus, rechtsradikale \textbar rechtsradikalen \textbar rechtsextreme, Heidenau \textbar Freital \textbar Bautzen, Ferguson \textbar Michael\_Brown \textbar Polizeigewalt} \\
  \midrule
  \multirow{2}{=}{Links\-ex\-tre\-mis\-mus} 
  & historisch & \underline{Ddr} \\
  & zeit\-ge\-nössisch & \underline{Linke} \\
  \midrule
  \multirow{2}{=}{Politischer Extremismus} 
  & historisch & \underline{Udssr \textbar Oktoberrevolution \textbar Stalins, Faschisten \textbar Nationalisten \textbar Kommunisten, Weimarer\_Republik \textbar Brd \textbar Ostberlin, Kommunismus \textbar Sozialismus \textbar Diktatur} \\
  & zeitgenössisch & \underline{Antifa \textbar Pegida-Bewegung \textbar Islam-} \\
  \lspbottomrule
 \end{tabularx}
\end{table}
\begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2014--2016}
\label{tab:2014_2}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{3}{=}{Islamismus} 
  & Nahostkonflikt & \underline{Ostjerusalem \textbar Israels\_Armee \textbar radikalislamischen, Hamas, Abbas \textbar Netanjahu \textbar Palästinensern, Gazastreifen \textbar Westjordanland, Jerusalem \textbar Tel\_Aviv \textbar Gaza, israelischen \textbar palästinensische \textbar israelische, Palästinenser \textbar Israelis} \\
  & Syrienkrieg / Islamischer Staat & \underline{Baschar\_al-Assad \textbar Machthaber\_Baschar\_al-Assad \textbar Assads, syrischen\_Rebellen \textbar IS-Dschihadisten, Assad, Regime \textbar Assad-regime \textbar militärisch, Al-Nusra-Front \textbar al-Qaida, IS-Miliz \textbar Isis, islamischen\_Staates \textbar iS, kurdische\_Kämpfer \textbar Peschmerga, Kämpfer \textbar Is-kämpfer, Kobane \textbar Mossul \textbar Kobani} \\
  & allgemein & \underline{Extremisten \textbar Dschihadisten \textbar Islamisten} \\
  \midrule
  variantenübergreifend & Diverses & \underline{mutmaßlichen\_Terroristen \textbar Abdelhamid\_Abaaoud \textbar Abaaoud} \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    15: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1717}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1408}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.6875}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Extremismus-Variante
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Bezug
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Frame-Elemente
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\multirow{2}{=}{Rechtsextremismus} & historisch & \underline{Nationalsozialisten\textbar Holocaust\textbar Nationalsozialismus (NS-Zeit)}

\underline{Judenverfolgung\textbar Vernichtungslager\textbar Konzentrationslager (NS-Zeit)} \\
& zeitgenössisch & \underline{Rechtsextremisten\textbar Rechtsextreme\textbar Neonazis}

\underline{rechtsradikalen\textbar rechtsextremen\textbar rechtsextreme (attributive Charakteristika Rechtsextremismus)}

\underline{Gauland\textbar Meuthen\textbar Weidel (NPD / AfD-Politiker*innen \& Wagenknecht)}

\underline{Uwe\_Böhnhardt\textbar Böhnhardt\textbar Uwe\_Mundlos (Böhnhardt / Mundlos / NSU)}

\underline{André\_E\textbar Nsu-prozess\textbar Beate\_Zschäpe (NSU)}

\underline{Antisemitismus\textbar Rassismus} \\
\multirow{2}{=}{Linksextremismus} & historisch & \\
& zeitgenössisch & \\
\multirow{2}{=}{Politischer Extremismus} & historisch & \underline{Ddr\textbar Weimarer\_Republik\textbar Sowjetunion (Totalitarismus / 20. Jhdt)}

\underline{Kommunismus\textbar Sozialismus\textbar Diktatur(Totalitarismus)}

\underline{Bolschewiki\textbar Kommunisten\textbar kommunistischen (Totalitarismus)} \\
& zeitgenössisch & \underline{Linkspartei\textbar Afd\textbar Linken (AfD / Linke / CDU)}

\underline{Rechtsradikalen\textbar Rechtsradikale\textbar Rechtsextremen (Akteure pol. Extremismus)} \\
\multirow{3}{=}{Islamismus} & Nahostkonflikt & \underline{Ostjerusalem\textbar Ramallah\textbar Hebron (Nahostkonflikt)}

\underline{Gazastreifen\textbar Westjordanland}

\underline{Hamas}

\underline{Palästinensern\textbar Israelis} \\
& Syrienkrieg / Islamischer Staat & \underline{Regierungstruppen\textbar Provinz\_Idlib\textbar syrischen\_Armee (Syrienkrieg)}

\underline{Idlib\textbar Afrin\textbar Damaskus (Orte in Syrien / Bagdad / Tripolis)}

\underline{Dschihadisten}

\underline{Terrormiliz\textbar Terrormiliz\_iS\textbar Terrormiliz\_islamischer\_Staat (IS und terroristische Gruppen)}

\underline{Pkk\textbar Terrororganisation\textbar Gülen-bewegung (PKK/Gülen-Bewegung/YPG)}

\underline{iS}

\underline{kurdische\textbar kurdischen\textbar syrische} \\
& Weiteres & \underline{Huthi-Rebellen\textbar Huthis\textbar Hisbollah ((para-)militärische Kräfte im Nahen Osten)}

\underline{Amri} \\
variantenübergreifend & Diverses & \underline{Rechtsextremismus\textbar Islamismus\textbar Extremismus (Extremismusformen / Radikalisierung / Fremdenfeindlichkeit)}

\underline{rechtsextremistischen\textbar gewaltbereiten\textbar militanten (gewalttätiger Extremismus)}

\underline{El\_Paso\textbar Christchurch\textbar Pittsburgh (rechtsextreme Anschläge in den USA / Amoklauf USA / isl. Anschlag Manchester)} \\
\end{longtable}

% \phantomsection\label{_Toc162477807}{}Tabelle : Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2017 -- 2019.
""",
        r"""\begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2017--2019}
\label{tab:2017}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{2}{=}{Rechts\-ex\-tre\-mis\-mus} 
  & historisch & \underline{Nationalsozialisten \textbar Holocaust \textbar Nationalsozialismus, Judenverfolgung \textbar Vernichtungslager \textbar Konzentrationslager} \\
  & zeitgenössisch & \underline{Rechtsextremisten \textbar Rechtsextreme \textbar Neonazis, rechtsradikalen \textbar rechtsextremen \textbar rechtsextreme, Gauland \textbar Meuthen \textbar Weidel, Uwe\_Böhnhardt \textbar Böhnhardt \textbar Uwe\_Mundlos, André\_E \textbar Nsu-prozess \textbar Beate\_Zschäpe, Antisemitismus \textbar Rassismus} \\
  \midrule
  \multirow{2}{=}{Links\-ex\-tre\-mis\-mus} 
  & historisch & - \\
  & zeitgenössisch & - \\
  \midrule
  \multirow{2}{=}{Politischer Extremismus} 
  & historisch & \underline{Ddr \textbar Weimarer\_Republik \textbar Sowjetunion, Kommunismus \textbar Sozialismus \textbar Diktatur, Bolschewiki \textbar Kommunisten \textbar kommunistischen} \\
  & zeitgenössisch & \underline{Linkspartei \textbar Afd \textbar Linken, Rechtsradikalen \textbar Rechtsradikale \textbar Rechtsextremen} \\
  \midrule
  \multirow{3}{=}{Islamismus} 
  & Nahostkonflikt & \underline{Ostjerusalem \textbar Ramallah \textbar Hebron, Gazastreifen \textbar Westjordanland, Hamas, Palästinensern \textbar Israelis} \\
  & Syrienkrieg / Islamischer Staat & \underline{Regierungstruppen \textbar Provinz\_Idlib \textbar syrischen\_Armee, Idlib \textbar Afrin \textbar Damaskus, Dschihadisten, Terrormiliz \textbar Terrormiliz\_iS \textbar Terrormiliz\_islamischer\_Staat, Pkk \textbar Terrororganisation \textbar Gülen-bewegung, iS, kurdische \textbar kurdischen \textbar syrische} \\
  & Weiteres & \underline{Huthi-Rebellen \textbar Huthis \textbar Hisbollah, Amri} \\
  \midrule
  variantenübergreifend & Diverses & \underline{Rechtsextremismus \textbar Islamismus \textbar Extremismus, rechtsextremistischen \textbar gewaltbereiten \textbar militanten, El\_Paso \textbar Christchurch \textbar Pittsburgh} \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    16: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1718}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1406}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.6876}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Extremismus-Variante
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Bezug
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Frame-Elemente
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\multirow{2}{=}{Rechtsextremismus} & historisch & \underline{Nationalsozialismus\textbar Holocaust}

\underline{Nazis} \\
& zeitgenössisch & \underline{Rechtsextremen\textbar Rechtsextremisten\textbar Neonazis (insb. rechte Akteure + Antifa)}

\underline{rechtsextremen\textbar antisemitische\textbar rechtsextreme (Attribute Rechtsextremismus / Chatgruppe / NSU)}

\underline{Höcke\textbar Kalbitz\textbar Andreas\_Kalbitz (AfD-Politiker / -Strukturen)}

\underline{Afd\textbar Partei}

\underline{Stephan\_Ernst\textbar Markus\_H\textbar Mittäter (Mord Walter Lübcke)}

\underline{Antisemitismus\textbar Rassismus\textbar Rechtsextremismus (Charakteristika Rechtsextrmeismus)}

\underline{George\_Floyd\textbar Floyd\textbar des\_Afroamerikaners\_George (Tod George Floyd)}

\underline{Portland\textbar Minneapolis\textbar Nationalgarde (Proteste Black Lives Matter)}

\underline{Hanau} \\
\multirow{2}{=}{Linksextremismus} & historisch & \underline{-} \\
& zeitgenössisch & \underline{-} \\
\multirow{2}{=}{Politischer Extremismus} & historisch & \underline{Mai\_1945\textbar Stalin\textbar Hitlers (20. Jhdt / Totalitarismus)}

\underline{1945\textbar1939\textbar1944 (Zweiter Weltkrieg / Totalitarismus)} \\
& zeitgenössisch & \underline{Susanne\_Hennig-Wellsow\textbar Hennig-Wellsow\textbar Mike\_Mohring (Politiker*innen, Wahl Thomas Kemmerich)} \\
\multirow{3}{=}{Islamismus} & Nahostkonflikt & \underline{Gaza\textbar Hamas\textbar Jerusalem (Nahostkonflikt)} \\
& Syrienkrieg / Islamischer Staat & \underline{Syrien\textbar Afghanistan\textbar Libyen}

\underline{iS\textbar Islamisten\textbar Terrormiliz\_islamischer\_Staat} \\
& Afghanistan & \underline{Taliban} \\
\end{longtable}

% \phantomsection\label{_Toc162477808}{}Tabelle : Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2020 -- 2021.
""",
        r"""\begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des Zeitraums 2020--2021}
\label{tab:2020}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{2}{=}{Rechts\-ex\-tre\-mis\-mus} 
  & historisch & \underline{Nationalsozialismus \textbar Holocaust, Nazis} \\
  & zeitgenössisch & \underline{Rechtsextremen \textbar Rechtsextremisten \textbar Neonazis, rechtsextremen \textbar antisemitische \textbar rechtsextreme, Höcke \textbar Kalbitz \textbar Andreas\_Kalbitz, Afd \textbar Partei, Stephan\_Ernst \textbar Markus\_H \textbar Mittäter, Antisemitismus \textbar Rassismus \textbar Rechtsextremismus, George\_Floyd \textbar Floyd \textbar des\_Afroamerikaners\_George, Portland \textbar Minneapolis \textbar Nationalgarde, Hanau} \\
  \midrule
  \multirow{2}{=}{Links\-ex\-tre\-mis\-mus} 
  & historisch & - \\
  & zeitgenössisch & - \\
  \midrule
  \multirow{2}{=}{Politischer Extremismus} 
  & historisch & \underline{Mai\_1945 \textbar Stalin \textbar Hitlers, 1945 \textbar 1939 \textbar 1944} \\
  & zeitgenössisch & \underline{Susanne\_Hennig-Wellsow \textbar Hennig-Wellsow \textbar Mike\_Mohring} \\
  \midrule
  \multirow{3}{=}{Islamismus} 
  & Nahostkonflikt & \underline{Gaza \textbar Hamas \textbar Jerusalem} \\
  & Syrienkrieg / Islamischer Staat & \underline{Syrien \textbar Afghanistan \textbar Libyen, iS \textbar Islamisten \textbar Terrormiliz\_islamischer\_Staat} \\
  & Afghanistan & \underline{Taliban} \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    17: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1717}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1408}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.6875}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Extremismus-Variante
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Bezug
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Frame-Elemente
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\multirow{2}{=}{Rechtsextremismus} & historisch & \underline{1945\textbar Kriegsende}

\underline{Nationalsozialismus}

\underline{Vernichtungslager\textbar Konzentrationslager\textbar Kz}

\underline{Deportationen\textbar Konzentrationslagern\textbar Deportation}

\underline{Wehrmacht\textbar Roten\_Armee\textbar Sowjets}

\underline{Nazis}

\underline{Holocaust}

\underline{Nazideutschland\textbar Nazi-deutschland\textbar nationalsozialistischen (NS-Zeit)}

\underline{Wehrmachtssoldaten\textbar Kz-häftlinge\textbar Deserteure (Wehrmacht / Kriegsopfer / Deserteure)}

\underline{Nazizeit\textbar Ns-zeit\textbar Weimarer\_Republik} \\
& zeitgenössisch & \underline{Rassisten\textbar Antisemiten\textbar Faschisten}

\underline{Afd\textbar Npd}

\underline{Dvu}

\underline{Alexander\_Gauland\textbar Jörg\_Meuthen\textbar Poggenburg (rechtsaußen Politiker)}

\underline{Schill\textbar Schill-Partei}

\underline{Nsu}

\underline{Hooligans\textbar Fußballfans\textbar Ultras}

\underline{Gauland\textbar Höcke\textbar Meuthen (AfD-Politiker / Möllemann)}

\underline{Zschäpe\textbar Beate\_Zschäpe}

\underline{rechtsradikale\textbar rechtsextreme\textbar antisemitische (Attribute Rechtsextremismus)}

\underline{Diskriminierung}

\underline{Islamophobie\textbar Xenophobie\textbar Islamfeindlichkeit (rechte Einstellungen / Schlagwörter)}

\underline{Antisemitismus\textbar Rassismus}

\underline{Rechtsextremisten\textbar Neonazis\textbar Rechtsextremen}

\underline{ausländerfeindliche\textbar fremdenfeindliche\textbar fremdenfeindlichen (Kennzeichen (insb. Rechts-)Extremismus} \\
\multirow{2}{=}{Linksextremismus} & historisch & \underline{Milošević}

\underline{Stasi\textbar Staatssicherheit} \\
& zeitgenössisch & \underline{linksradikalen\textbar marxistischen\textbar anarchistischen (Attribute Strömungen des Linksextresmismus)}

\underline{Pds\textbar Linkspartei}

\underline{Antifa-gruppen\textbar Aktivistinnen\textbar linke\_Gruppen ((insb. linke) aktivistische Gruppen)}

\underline{Linke}

\underline{Gegendemonstranten\textbar Randalierer\textbar Einsatzkräfte}

\underline{Oskar\_Lafontaine\textbar Lafontaine\textbar Gregor\_Gysi (Politiker*innen Linkspartei)}

\underline{Heiligendamm\textbar Genua\textbar G-8-Gipfel}

\underline{gegen\_Rechts\textbar antifaschistische\textbar Antifa} \\
\multirow{2}{=}{Politischer Extremismus} & historisch & \underline{Kpd\textbar Nsdap\textbar Sed}

\underline{Kommunismus\textbar Sozialismus\textbar Faschismus}

\underline{Hitler\textbar Adolf\_Hitler\textbar Stalin ermordeten\_Juden\textbar Reichspogromnacht\textbar Ns-opfer (NS- \& DDR-Zeit)} \\
& zeitgenössisch & \underline{Rechtspopulismus\textbar Populisten\textbar Populismus (Populismus / Nationalismus / PEGIDA)}

\underline{rechten\_Szene\textbar rechtsextremen\_Szene\textbar linken\_Szene}

\underline{Antifas\textbar Skins\textbar Antifaschisten (Akteure Linksextremismus / Skinheads / Rechtsradikalismus)}

\underline{Npd-kundgebung\textbar Neonazi-Demo\textbar NPD-Demo (politische Demonstrationen / Aktivismus)} \\
\multirow{4}{=}{Islamismus} & Nahostkonflikt & \underline{jüdischen\_Siedlungen\textbar Palästinensergebiete\textbar Golanhöhen (Nahostkonflikt)}

\underline{Arbeitspartei\textbar Ehud\_Barak\textbar Benjamin\_Netanjahu (Politiker Nahostkonflikt)}

\underline{Palästinenserpräsident\_Mahmud\_Abbas\textbar Palästinenserpräsident\_Jassir\_Arafat\textbar Jassir\_Arafat (pol. Akteure Nahostkonflikt)}

\underline{iS}

\underline{Israel\textbar Palästina}

\underline{israelische\textbar palästinensische}

\underline{israelischen\textbar palästinensischen}

\underline{Palästinenser\textbar Israelis}

\underline{Ramallah\textbar Jerusalem\textbar Gaza}

\underline{Scharon\textbar Arafat\textbar Barak (Politiker Nahostkonflikt)}

\underline{Gaza-streifen\textbar Gazastreifen\textbar Westjordanland}

\underline{Hebron\textbar Nablus\textbar Dschenin (Nahostkonflikt)}

\underline{Palästinenserpräsident\_Arafat\textbar Präsident\_Mahmud\_Abbas\textbar Abu\_Masen (Nahostkonflikt)} \\
& Kriege Naher / Mittlerer Osten & \underline{Mossul\textbar Falludscha\textbar Homs (Städte Irak / Syrien./ Afghanistan)}

\underline{Syrien\textbar Libyen\textbar Afghanistan}

\underline{Huthis\textbar Baschar\_al-Assad\textbar Sadr (pol. Akteure Naher / Mittlerer Osten)}

\underline{Iraks\textbar Libyens\textbar Syriens (Länder Naher / Mittlerer Osten)}

\underline{syrischen\textbar libanesischen\textbar irakischen (Adjektive Länder Naher / Mittlerer Osten)}

\underline{syrische\textbar irakische\textbar iranische (Adjektive Länder Naher / Mittlerer Osten)} \\
& Terrorismus & \underline{Bin\_Laden\textbar Ussama\_Bin\_Laden\textbar al-Qaida}

\underline{Hisbollah\textbar Hamas\textbar Pkk}

\underline{Al-kaida\textbar Al-Qaida\textbar Isis (Terror. Akteure Islamismus)}

\underline{mutmaßlichen\_Terroristen\textbar Al-Qaida-Mitglieder\textbar Bombenleger ({[}insb. islamistische{]}terroristische Akteure)} \\
& Weiteres & \underline{Mursi\textbar Chatami\textbar Mubarak (führende Politiker Naher und Mittlerer Osten)}

\underline{Saddam\_Hussein\textbar Gaddafi\textbar Saddam (dikt. Staatsoberhäupter Naher / Mittlerer Osten)}

\underline{Extremisten\textbar Islamisten\textbar Dschihadisten}

\underline{schiitischen\textbar sunnitischen\textbar sunnitische (schiitisch / sunnitisch / islamistisch)}

\underline{Muslimbrüder\textbar Radikalen\textbar Muslimbruderschaft (Akteure des Islamismus)} \\
& & \underline{Gorleben}

\underline{Rechtsradikalismus\textbar Fremdenfeindlichkeit\textbar Extremismus (Extremismus)} \\
\end{longtable}

% \phantomsection\label{_Ref162017445}{}Tabelle : Thematisch gruppierte Auswahl semantisch äquivalenter Cluster der \textsc{TAZ}.
""",
        r"""\begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster der \textsc{TAZ}}
\label{tab:taz_1}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{2}{=}{Rechts\-ex\-tre\-mis\-mus} 
  & historisch & \underline{1945 \textbar Kriegsende, Nationalsozialismus, Vernichtungslager \textbar Konzentrationslager \textbar Kz, Deportationen \textbar Konzentrationslagern \textbar Deportation, Wehrmacht \textbar Roten\_Armee \textbar Sowjets, Nazis, Holocaust, Nazideutschland \textbar Nazi-deutschland \textbar nationalsozialistischen, Wehrmachtssoldaten \textbar Kz-häftlinge \textbar Deserteure, Nazizeit \textbar Ns-zeit \textbar Weimarer\_Republik} \\
  & zeitgenössisch & \underline{Rassisten \textbar Antisemiten \textbar Faschisten, Afd \textbar Npd, Dvu, Alexander\_Gauland \textbar Jörg\_Meuthen \textbar Poggenburg, Schill \textbar Schill-Partei, Nsu, Hooligans \textbar Fußballfans \textbar Ultras, Gauland \textbar Höcke \textbar Meuthen, Zschäpe \textbar Beate\_Zschäpe, rechtsradikale \textbar rechtsextreme \textbar antisemitische, Diskriminierung, Islamophobie \textbar Xenophobie \textbar Islamfeindlichkeit, Antisemitismus \textbar Rassismus, Rechtsextremisten \textbar Neonazis \textbar Rechtsextremen, ausländerfeindliche \textbar fremdenfeindliche \textbar fremdenfeindlichen} \\
  \lspbottomrule
 \end{tabularx}
\end{table}

 \begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster der \textsc{TAZ}}
\label{tab:taz_2}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{2}{=}{Links\-ex\-tre\-mis\-mus} 
  & historisch & \underline{Milošević, Stasi \textbar Staatssicherheit} \\
  & zeitgenössisch & \underline{linksradikalen \textbar marxistischen \textbar anarchistischen, Pds \textbar Linkspartei, Antifa-gruppen \textbar Aktivistinnen \textbar linke\_Gruppen, Linke, Gegendemonstranten \textbar Randalierer \textbar Einsatzkräfte, Oskar\_Lafontaine \textbar Gregor\_Gysi, Heiligendamm \textbar Genua \textbar G-8-Gipfel, gegen\_Rechts \textbar antifaschistische \textbar Antifa} \\
  \midrule
  \multirow{2}{=}{Politischer Extremismus} 
  & historisch & \underline{Kpd \textbar Nsdap \textbar Sed, Kommunismus \textbar Sozialismus \textbar Faschismus, Hitler \textbar Adolf\_Hitler \textbar Stalin, ermordeten\_Juden \textbar Reichspogromnacht \textbar Ns-opfer} \\
  & zeitgenössisch & \underline{Rechtspopulismus \textbar Populisten \textbar Populismus, rechten\_Szene \textbar rechtsextremen\_Szene \textbar linken\_Szene, Antifas \textbar Skins \textbar Antifaschisten, Npd-kundgebung \textbar Neonazi-Demo \textbar NPD-Demo} \\
  \lspbottomrule
 \end{tabularx}
\end{table}

 \begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster der \textsc{TAZ}}
\label{tab:taz_3}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{4}{=}{Islamismus} 
  & Nahostkonflikt & \underline{jüdischen\_Siedlungen \textbar Palästinensergebiete \textbar Golanhöhen, Arbeitspartei \textbar Ehud\_Barak \textbar Benjamin\_Netanjahu, Palästinenserpräsident\_Mahmud\_Abbas \textbar Palästinenserpräsident\_Jassir\_Arafat \textbar Jassir\_Arafat, iS, Israel \textbar Palästina, israelische \textbar palästinensische, israelischen \textbar palästinensischen, Palästinenser \textbar Israelis, Ramallah \textbar Jerusalem \textbar Gaza, Scharon \textbar Arafat \textbar Barak, Gaza-streifen \textbar Gazastreifen \textbar Westjordanland, Hebron \textbar Nablus \textbar Dschenin, Palästinenserpräsident\_Arafat \textbar Präsident\_Mahmud\_Abbas \textbar Abu\_Masen} \\
  & Kriege Naher / Mittlerer Osten & \underline{Mossul \textbar Falludscha \textbar Homs, Syrien \textbar Libyen \textbar Afghanistan, Huthis \textbar Baschar\_al-Assad \textbar Sadr, Iraks \textbar Libyens \textbar Syriens, syrischen \textbar libanesischen \textbar irakischen, syrische \textbar irakische \textbar iranische} \\
  & Terrorismus & \underline{Bin\_Laden \textbar Ussama\_Bin\_Laden \textbar al-Qaida, Hisbollah \textbar Hamas \textbar Pkk, Al-kaida \textbar Al-Qaida \textbar Isis, mutmaßlichen\_Terroristen \textbar Al-Qaida-Mitglieder \textbar Bombenleger} \\
  & Weiteres & \underline{Mursi \textbar Chatami \textbar Mubarak, Saddam\_Hussein \textbar Gaddafi \textbar Saddam, Extremisten \textbar Islamisten \textbar Dschihadisten, schiitischen \textbar sunnitischen \textbar sunnitische, Muslimbrüder \textbar Radikalen \textbar Muslimbruderschaft} \\
  \midrule
  variantenübergreifend & & \underline{Gorleben, Rechtsradikalismus \textbar Fremdenfeindlichkeit \textbar Extremismus} \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    18: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1717}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1408}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.6875}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Extremismus-Variante
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Bezug
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Frame-Elemente
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\multirow{2}{=}{Rechtsextremismus} & historisch & \underline{Nationalsozialisten\textbar Nazis\textbar Nationalsozialismus}

\underline{Konzentrationslager\textbar Vernichtungslager\textbar Auschwitz (NS-Zeit)}

\underline{Nazizeit\textbar Ns-zeit\textbar Nazi-zeit (NS-Zeit / RAF)} \\
& zeitgenössisch & \underline{Afd\textbar Npd}

\underline{Gauland\textbar Petry\textbar Höcke (AfD-Politiker*innen / Sarrazin / Wagenknecht / Pauli)}

\underline{Neonazis\textbar Rechtsextremisten\textbar Rechtsextreme}

\underline{rechtsradikalen\textbar rechtsradikale\textbar rechtsextremen (attribute Rechtsextremismus)}

\underline{Beate\_Zschäpe\textbar Zschäpe\textbar Nsu-prozess}

\underline{Uwe\_Böhnhardt\textbar Böhnhardt\textbar Uwe\_Mundlos}

\underline{rechtspopulistischen\textbar Front\_national\textbar rechtspopulistische (Rechtspopulismus / rechte Parteien Europa)}

\underline{Rechtsextremismus}

\underline{Rassismus\textbar Antisemitismus} \\
\multirow{2}{=}{Linksextremismus} & historisch & \underline{Ddr}

\underline{Brd\textbar friedlichen\_Revolution\textbar Walter\_Ulbricht (Totalitarismus)}

\underline{Udssr\textbar Sowjetunion\textbar sowjetischen (Kalter Krieg)}

\underline{Stasi} \\
& zeitgenössisch & \underline{Linkspartei\textbar Pds\textbar Linken}

\underline{G-8-Gipfel\textbar G7-Gipfel\textbar G-20-Gipfel (internationale Gipfel)}

\underline{Lafontaine\textbar Gysi\textbar Oskar\_Lafontaine (Politiker Linkspartei / Möllemann / Wowereit)} \\
\multirow{2}{=}{Politischer Extremismus} & historisch & \underline{Hitler\textbar Adolf\_Hitler\textbar Stalin} \\
& zeitgenössisch & \\
\multirow{4}{=}{Islamismus} & Nahostkonflikt & \underline{Rafah\textbar israelische\_Panzer\textbar Nablus (Nahostkonflikt)}

\underline{Scharon\textbar Netanjahu\textbar Barak (Politiker Nahostkonflikt)}

\underline{Gaza-streifen\textbar Gazastreifen\textbar Westjordanland}

\underline{Israels\_Ministerpräsident\_Ariel\_Scharon\textbar Palästinenser-Präsident\_Mahmud\_Abbas\textbar Palästinenser-Präsident\_Jassir\_Arafat (Politiker Nahostkonflikt)}

\underline{Gaza\textbar Ramallah}

\underline{Ostjerusalem\textbar besetzten\_Gebieten\textbar Ost-jerusalem (Orte Nahostkonflikt)}

\underline{Palästinenserpräsident\_Mahmud\_Abbas\textbar Palästinenserpräsident\_Jassir\_Arafat\textbar palästinensischen\_Autonomiebehörde (pol. Akteure Nahostkonflikt)}

\underline{Palästinenser\textbar Israelis}

\underline{palästinensische\textbar israelischen\textbar israelische} \\
& Kriege Naher / Mittlerer Osten & \underline{Falludscha\textbar Homs\textbar Mossul (Orte Naher / Mittlerer Osten)}

\underline{Damaskus}

\underline{Dschalalabad\textbar Hauptstadt\_Bagdad\textbar Kirkuk (Orte Naher / Mittlerer Osten)}

\underline{Libanon\textbar Jemen\textbar Nachbarland}

\underline{Salih\textbar Husni\_Mubarak\textbar Maliki (pol. Akteure Naher / Mittlerer Osten)}

\underline{Syrien\textbar Libyen\textbar Afghanistan}

\underline{Saddam\_Hussein\textbar Gaddafi\textbar Saddam (Diktatoren Naher / Mittlerer Osten)}

\underline{Saudi-arabien\textbar Ägypten\textbar Pakistan (Länder Naher / Mittlerer Osten)}

\underline{irakische\textbar syrische\textbar libanesische (Adjektive Länder Naher / Mittlerer Osten)}

\underline{syrischen\textbar irakischen\textbar libanesischen (Adjektive Länder Naher / Mittlerer Osten)}

\underline{Assads\textbar Baschar\_al-Assad\textbar Präsident\_Baschar\_al-Assad (Diktatoren Naher / Mittlerer Osten / Chemiewaffen)}

\underline{Bagdad\textbar Kabul}

\underline{Iraker\textbar Afghanen\textbar Syrer}

\underline{Syriens\textbar Libyens\textbar Saudi-arabiens (Länder Naher./ Mittlerer Osten)}

\underline{Norden\_Syriens\textbar Nordirak\textbar syrischen\_Bürgerkrieg (Orte Naher / Mittlerer Osten / Kriegsgebiete)} \\
& Terrorismus & \underline{Sarkawi\textbar Osama\_bin\_Laden\textbar Abu\_Mussab\_al-Sarkawi (Akteure islamistischer Terrorismus)}

\underline{Terrormiliz\textbar IS-Miliz\textbar syrischen\_Rebellen (militärische Akteure Naher / Mittlerer Osten)}

\underline{Amri}

\underline{Taliban}

\underline{Osama\_Bin\_Laden\textbar Bin\_Laden\textbar al-Qaida}

\underline{Terrororganisation\_al-Qaida\textbar Qaida\textbar Terrorgruppe}

\underline{Hisbollah\textbar Hamas\textbar Pkk} \\
& Weiteres & \underline{schiitische\textbar sunnitischen\textbar schiitischen (Attribute Islamismus)}

\underline{Extremisten\textbar Islamisten\textbar Dschihadisten}

\underline{Intoleranz\textbar Fremdenfeindlichkeit\textbar Ausländerfeindlichkeit (Charakteristika Extremismus / Islamismus)} \\
variantenübergreifend & & \underline{Zwickauer\_Zelle\textbar Rechtsterroristen\textbar Zwickauer\_Terrorzelle (Akteure Terrorismus aller Extremismusvarianten)} \\
\end{longtable}

% \phantomsection\label{_Toc162477810}{}Tabelle : Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des \textsc{Spiegel}.
""",
        r"""\begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des \textsc{Spiegel}}
\label{tab:spiegel_1}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{2}{=}{Rechts\-ex\-tre\-mis\-mus} 
  & historisch & \underline{Nationalsozialisten \textbar Nazis \textbar Nationalsozialismus, Konzentrationslager \textbar Vernichtungslager \textbar Auschwitz, Nazizeit \textbar Ns-zeit \textbar Nazi-zeit} \\
  & zeitgenössisch & \underline{Afd \textbar Npd, Gauland \textbar Petry \textbar Höcke, Neonazis \textbar Rechtsextremisten \textbar Rechtsextreme, rechtsradikalen \textbar rechtsradikale \textbar rechtsextremen, Beate\_Zschäpe \textbar Nsu-prozess, Uwe\_Böhnhardt \textbar Uwe\_Mundlos, rechtspopulistischen \textbar Front\_national \textbar rechtspopulistische, Rechtsextremismus, Rassismus \textbar Antisemitismus} \\
  \midrule
  \multirow{2}{=}{Links\-ex\-tre\-mis\-mus} 
  & historisch & \underline{Ddr, Brd \textbar friedlichen\_Revolution \textbar Walter\_Ulbricht, Udssr \textbar Sowjetunion \textbar sowjetischen, Stasi} \\
  & zeitgenössisch & \underline{Linkspartei \textbar Pds \textbar Linken, G-8-Gipfel \textbar G7-Gipfel \textbar G-20-Gipfel, Lafontaine \textbar Gysi \textbar Oskar\_Lafontaine} \\
  \midrule
  \multirow{2}{=}{Politischer Extremismus} 
  & historisch & \underline{Hitler \textbar Adolf\_Hitler \textbar Stalin} \\
  & zeitgenössisch & - \\
  \midrule
  variantenübergreifend & & \underline{Zwickauer\_Zelle \textbar Rechtsterroristen \textbar Zwickauer\_Terrorzelle} \\
  \lspbottomrule
 \end{tabularx}
\end{table}

\begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster des \textsc{Spiegel}}
\label{tab:spiegel_2}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{4}{=}{Islamismus} 
  & Nahostkonflikt & \underline{Rafah \textbar israelische\_Panzer \textbar Nablus, Scharon \textbar Netanjahu \textbar Barak, Gaza-streifen \textbar Gazastreifen \textbar Westjordanland, Israels\_Ministerpräsident\_Ariel\_Scharon \textbar Palästinenser-Präsident\_Mahmud\_Abbas \textbar Palästinenser-Präsident\_Jassir\_Arafat, Gaza \textbar Ramallah, Ostjerusalem \textbar besetzten\_Gebieten \textbar Ost-jerusalem, Palästinenserpräsident\_Mahmud\_Abbas \textbar Palästinenserpräsident\_Jassir\_Arafat \textbar palästinensischen\_Autonomiebehörde, Palästinenser \textbar Israelis, palästinensische \textbar israelischen \textbar israelische} \\
  & Kriege Naher / Mittlerer Osten & \underline{Falludscha \textbar Homs \textbar Mossul, Damaskus, Dschalalabad \textbar Hauptstadt\_Bagdad \textbar Kirkuk, Libanon \textbar Jemen \textbar Nachbarland, Salih \textbar Husni\_Mubarak \textbar Maliki, Syrien \textbar Libyen \textbar Afghanistan, Saddam\_Hussein \textbar Gaddafi \textbar Saddam, Saudi-arabien \textbar Ägypten \textbar Pakistan, irakische \textbar syrische \textbar libanesische, syrischen \textbar irakischen \textbar libanesischen, Assads \textbar Baschar\_al-Assad \textbar Präsident\_Baschar\_al-Assad, Bagdad \textbar Kabul, Iraker \textbar Afghanen \textbar Syrer, Syriens \textbar Libyens \textbar Saudi-arabiens, Norden\_Syriens \textbar Nordirak \textbar syrischen\_Bürgerkrieg} \\
  & Terrorismus & \underline{Sarkawi \textbar Osama\_bin\_Laden \textbar Abu\_Mussab\_al-Sarkawi, Terrormiliz \textbar IS-Miliz \textbar syrischen\_Rebellen, Amri, Taliban, Osama\_Bin\_Laden \textbar al-Qaida, Terrororganisation\_al-Qaida \textbar Qaida \textbar Terrorgruppe, Hisbollah \textbar Hamas \textbar Pkk} \\
  & Weiteres & \underline{schiitische \textbar sunnitischen \textbar schiitischen, Extremisten \textbar Islamisten \textbar Dschihadisten, Intoleranz \textbar Fremdenfeindlichkeit \textbar Ausländerfeindlichkeit} \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    19: (
        r"""\begin{longtable}[]{@{}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1717}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.1408}}
  >{\raggedright\arraybackslash}p{(\linewidth - 4\tabcolsep) * \real{0.6875}}@{}}
\toprule\noalign{}
\begin{minipage}[b]{\linewidth}\raggedright
Extremismus-Variante
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Bezug
\end{minipage} & \begin{minipage}[b]{\linewidth}\raggedright
Frame-Elemente
\end{minipage} \\
\midrule\noalign{}
\endhead
\bottomrule\noalign{}
\endlastfoot
\multirow{2}{=}{Rechtsextremismus} & historisch & \underline{Juli\_1944\textbar Nazi-deutschland\textbar Stalingrad (2. Weltkrieg)}

\underline{Vernichtungslager\textbar Konzentrationslagern\textbar Konzentrationslager (KZs / Srebenica / SED-Diktatur)}

\underline{Ns-diktatur\textbar Nazi-diktatur\textbar Judenverfolgung (Krieg)}

\underline{Holocaust\textbar Nationalsozialismus}

\underline{Hitler}

\underline{1945\textbar Kriegsende}

\underline{Ss\textbar Hitlers\textbar Wehrmacht (Nazi-Akteure NS-Zeit / Auschwitz)}

\underline{Nazis\textbar Nationalsozialisten}

\underline{Nazizeit\textbar Weimarer\_Republik\textbar des\_Dritten\_Reiches (historische Umbrüche)} \\
& zeitgenössisch & \underline{Beate\_Zschäpe\textbar Zschäpe\textbar Nsu}

\underline{Uwe\_Böhnhardt\textbar Uwe\_Mundlos\textbar Böhnhardt}

\underline{Ultras\textbar Hooligans\textbar Fußballfans}

\underline{Rechtsterroristen\textbar Hintermänner\textbar mutmaßlichen\_Terroristen (NSU / Ermittlungen)}

\underline{Fremdenfeindlichkeit\textbar Intoleranz\textbar Ausländerfeindlichkeit (Charakteristika Rechtsextremismus)}

\underline{antisemitische\textbar rassistische\textbar antisemitischen (attributive Charakteristika Rechtsextremismus)}

\underline{Antisemitismus\textbar Rassismus\textbar Rechtsextremismus}

\underline{Rechtsextremisten\textbar Neonazis\textbar Rechtsextremen (Akteure Rechtsextremismus / Salafisten)}

\underline{Hells\_Angels\textbar Bandidos\textbar Rocker} \\
\multirow{2}{=}{Linksextremismus} & historisch & \underline{Ddr}

\underline{Erich\_Honecker\textbar Walter\_Ulbricht\textbar Honecker (DDR-Zeit)}

\underline{Stasi\textbar Staatssicherheit\textbar Sed}

\underline{kommunistischen\textbar kommunistische\textbar sozialistischen}

\underline{Sowjetunion\textbar Udssr\textbar sowjetischen (Kalter Krieg)} \\
& zeitgenössisch & \underline{G-20-Gipfel\textbar G-8-Gipfel\textbar G-7-gipfel (Internationale Gipfeltreffen)} \\
\multirow{2}{=}{Politischer Extremismus} & historisch & \underline{Nationalismus\textbar Fundamentalismus\textbar Totalitarismus (Ideologien)}

\underline{Sozialismus\textbar Kommunismus\textbar Kapitalismus} \\
& zeitgenössisch & \underline{rechtsradikalen\textbar rechtsextremistischen\textbar linksextremen (politischer Extremismus)}

\underline{sozialdemokratische\textbar sozialdemokratischen\textbar Gregor\_Gysi (sozialdemokratisch / Gysi / Sarrazin)}

\underline{linken\textbar rechte\textbar linke}

\underline{Gauland\textbar Meuthen\textbar Petry (politische Akteure AfD / Gysi / Wagenknecht / Möllemann / Sarrazin / Özdemir)}

\underline{Linkspartei\textbar Pds\textbar Linken (Linkspartei / PDS / AfD)}

\underline{nationalistischen\textbar nationalistische\textbar Nationalisten (Nationalismus / Populismus / Radikalismus)} \\
\multirow{4}{=}{Islamismus} & Nahostkonflikt & \underline{Peres\textbar Ariel\_Scharon\textbar Palästinenserpräsident\_Mahmud\_Abbas (Politiker Nahostkonflikt)}

\underline{Hebron\textbar Rafah\textbar Nablus (Orte Nahost)}

\underline{Gazastreifen\textbar Westjordanland}

\underline{palästinensischen\_Autonomiebehörde\textbar Palästinensergebiete\textbar radikalislamische\_Hamas (Nahostkonflikt)}

\underline{Palästinenser\textbar Israelis}

\underline{Scharon\textbar Abbas\textbar Arafat (Politiker Nahostkonflikt)} \\
& Kriege Naher Osten & \underline{Syrien\textbar Afghanistan\textbar Libyen}

\underline{iranische\textbar iranischen\textbar israelische}

\underline{Falludscha\textbar Raka\textbar syrischen\_Armee (Krieg Naher Osten)}

\underline{Saddam\_Hussein\textbar Saddam\textbar Gaddafi (Diktatoren Naher Osten)}

\underline{Jemen\textbar Libanon}

\underline{Syriens\textbar Saudi-arabiens\textbar Irans (Länder Naher / Mittlerer Osten)}

\underline{palästinensischen\textbar israelischen\textbar syrischen}

\underline{Bagdad\textbar Kabul}

\underline{Annan\textbar Un-generalsekretär\_Ban\_Ki\_Moon\textbar syrische\_Opposition}

\underline{Syrer\textbar Iraker\textbar Afghanen}

\underline{Damaskus\textbar Aleppo} \\
& Terrorismus & \underline{al-Qaida\textbar Al\_Qaida\textbar Bin\_Laden}

\underline{iS}

\underline{Terrormiliz\textbar Al-kaida\textbar Terrormiliz\_iS}

\underline{irakische\textbar irakischen\textbar syrische}

\underline{Hisbollah\textbar Hamas\textbar Pkk} \\
& Diverses & \underline{Islamisten}

\underline{Terrorist\textbar Islamist\textbar Mörder}

\underline{Extremisten\textbar Dschihadisten\textbar Terroristen}

\underline{militante\textbar islamistische\textbar terroristische Terrorgruppe\textbar Terrororganisation\textbar Terrormiliz\_islamischer\_Staat (IS / RAF)} \\
\end{longtable}

% \phantomsection\label{_Toc162477811}{}Tabelle : Thematisch gruppierte Auswahl semantisch äquivalenter Cluster der \textsc{Welt}.
""",
        r"""\begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster der \textsc{Welt}}
\label{tab:welt_1}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{2}{=}{Rechts\-ex\-tre\-mis\-mus} 
  & historisch & \underline{Juli\_1944 \textbar Nazi-deutschland \textbar Stalingrad, Vernichtungslager \textbar Konzentrationslagern \textbar Konzentrationslager, Ns-diktatur \textbar Nazi-diktatur \textbar Judenverfolgung, Holocaust \textbar Nationalsozialismus, Hitler, 1945 \textbar Kriegsende, Ss \textbar Hitlers \textbar Wehrmacht, Nazis \textbar Nationalsozialisten, Nazizeit \textbar Weimarer\_Republik \textbar des\_Dritten\_Reiches} \\
  & zeitgenössisch & \underline{Beate\_Zschäpe \textbar Nsu, Uwe\_Böhnhardt \textbar Uwe\_Mundlos \textbar Böhnhardt, Ultras \textbar Hooligans \textbar Fußballfans, Rechtsterroristen \textbar Hintermänner \textbar mutmaßlichen\_Terroristen, Fremdenfeindlichkeit \textbar Intoleranz \textbar Ausländerfeindlichkeit, antisemitische \textbar rassistische \textbar antisemitischen, Antisemitismus \textbar Rassismus \textbar Rechtsextremismus, Rechtsextremisten \textbar Neonazis \textbar Rechtsextremen, Hells\_Angels \textbar Bandidos \textbar Rocker} \\
  \midrule
  \multirow{2}{=}{Links\-ex\-tre\-mis\-mus} 
  & historisch & \underline{Ddr, Erich\_Honecker \textbar Walter\_Ulbricht \textbar Honecker, Stasi \textbar Staatssicherheit \textbar Sed, kommunistischen \textbar kommunistische \textbar sozialistischen, Sowjetunion \textbar Udssr \textbar sowjetischen} \\
  & zeitgenössisch & \underline{G-20-Gipfel \textbar G-8-Gipfel \textbar G-7-gipfel} \\
  \midrule
  \multirow{2}{=}{Politischer Extremismus} 
  & historisch & \underline{Nationalismus \textbar Fundamentalismus \textbar Totalitarismus, Sozialismus \textbar Kommunismus \textbar Kapitalismus} \\
  & zeitgenössisch & \underline{rechtsradikalen \textbar rechtsextremistischen \textbar linksextremen, sozialdemokratische \textbar sozialdemokratischen \textbar Gregor\_Gysi, linken \textbar rechte \textbar linke, Gauland \textbar Meuthen \textbar Petry, Linkspartei \textbar Pds \textbar Linken, nationalistischen \textbar nationalistische \textbar Nationalisten} \\
  \lspbottomrule
 \end{tabularx}
\end{table}

\begin{table}
\caption{Thematisch gruppierte Auswahl semantisch äquivalenter Cluster der \textsc{Welt}}
\label{tab:welt_2}
 \begin{tabularx}{\textwidth}{>{\hsize=0.5\hsize}X >{\hsize=0.5\hsize}X >{\hsize=2.0\hsize}X}
  \lsptoprule
  Extremismus-Variante & Bezug & Frame-Elemente \\
  \midrule
  \multirow{4}{=}{Islamismus} 
  & Nahostkonflikt & \underline{Peres \textbar Ariel\_Scharon \textbar Palästinenserpräsident\_Mahmud\_Abbas, Hebron \textbar Rafah \textbar Nablus, Gazastreifen \textbar Westjordanland, palästinensischen\_Autonomiebehörde \textbar Palästinensergebiete \textbar radikalislamische\_Hamas, Palästinenser \textbar Israelis, Scharon \textbar Abbas \textbar Arafat} \\
  & Kriege Naher Osten & \underline{Syrien \textbar Afghanistan \textbar Libyen, iranische \textbar iranischen \textbar israelische, Falludscha \textbar Raka \textbar syrischen\_Armee, Saddam\_Hussein \textbar Saddam \textbar Gaddafi, Jemen \textbar Libanon, Syriens \textbar Saudi-arabiens \textbar Irans, palästinensischen \textbar israelischen \textbar syrischen, Bagdad \textbar Kabul, Annan \textbar Un-generalsekretär\_Ban\_Ki\_Moon \textbar syrische\_Opposition, Syrer \textbar Iraker \textbar Afghanen, Damaskus \textbar Aleppo} \\
  & Terrorismus & \underline{al-Qaida \textbar Al\_Qaida \textbar Bin\_Laden, iS, Terrormiliz \textbar Al-kaida \textbar Terrormiliz\_iS, irakische \textbar irakischen \textbar syrische, Hisbollah \textbar Hamas \textbar Pkk} \\
  & Diverses & \underline{Islamisten, Terrorist \textbar Islamist \textbar Mörder, Extremisten \textbar Dschihadisten \textbar Terroristen, militante \textbar islamistische \textbar terroristische Terrorgruppe \textbar Terrororganisation \textbar Terrormiliz\_islamischer\_Staat} \\
  \lspbottomrule
 \end{tabularx}
\end{table}""",
    ),
    #20: (r"""""", r""""""),
}

In [6]:
for key, value in tables.items():
    tables[key] = (
        value[0],
        value[1].replace(
            "& zeitgenössisch", "& zeit\-ge\-nössisch"
            ).replace(
                "variantenübergreifend &", "varianten\-über\-greifend &"
            ).replace(
                "Nahostkonflikt &", "Nahost\-konflikt &"
            ).replace(
                "Linksextremismus &", "Links\-ex\-tre\-mis\-mus &"
            ).replace(
                "außerparlamentarisch &", "außer\-par\-la\-men\-ta\-risch &"
            )
        )

In [7]:
import re

def extract_longtable_rows(latex):
    # 1) This regex finds one row = one minipage cell + three &‐separated cells + final \\
    row_re = re.compile(r'''
      \\begin\{minipage\}\[t\]\{[^\}]+\}\\raggedright  # col 1 begins
      \s*
      (?P<sigle>K\d{2})\\*\s*                         # e.g. K20 ⏎
      (?P<name>[^\n]+)\\\\\s*                          # e.g. spiegel_ ⏎
      (?P<num>\d+)\s*\\strut                         # e.g. 1566482\strut
      .*?\\end\{minipage\}\s*&\s*                    # end col1 + &
      (?P<left>.*?)\s*&\s*                           # col2 = left context
      (?P<match>.*?)\s*&\s*                          # col3 = the two clusters + conj
      (?P<right>.*?)\s*\\\\                          # col4 = right context + \\
      ''', re.DOTALL | re.VERBOSE)

    results = []
    for m in row_re.finditer(latex):
        # reconstruct text_id
        sigle   = m.group('sigle')
        name    = m.group('name').strip()  # remove surrounding whitespace
        num     = m.group('num')
        text_id = f"{name}_{num}"

        # clean up contexts
        left    = re.sub(r'\s+', ' ', m.group('left')).strip()
        right   = re.sub(r'\s+', ' ', m.group('right')).strip()
        raw     = m.group('match').strip()

        # 2) extract clusters as anything between "{[}" and "{]}"
        clusters = re.findall(r'(\{\[}.*?\{]\})', raw, re.DOTALL)
        if len(clusters) < 2:
            # If you expect rows that might not have two clusters, 
            # you might want to handle this more gracefully, e.g., by logging and skipping the row.
            raise ValueError(f"Expected ≥2 clusters in row {sigle}, got {clusters!r}")

        first_match = clusters[0].strip()
        third_match = clusters[1].strip()

        # 3) remove clusters from raw so that only the conjunction remains
        tmp = raw
        for cl in clusters:
            tmp = tmp.replace(cl, ' ')
            
        # 4) find the leftover \textbf{...}  →  the conjunction(s)
        #    Multiple bolded parts are now allowed and will be joined.
        bolds = re.findall(r'(\\textbf\{.*?\})', tmp)
        second_match = ' '.join(bolds) # Join all found bold parts with a space

        results.append({
            'concordance_sigle': sigle,
            'text_id':           text_id,
            'left_context':      left,
            'first_match':       first_match,
            'second_match':      second_match,
            'third_match':       third_match,
            'right_context':     right
        })

    return results

# make_grouped_table and comment_concordances functions remain the same as your last correct version.
# I'll include them here for completeness.

def make_grouped_table(results, caption="Tabularx mit 3 Spalten"):
    """
    Given `results` = list of dicts with keys
      ['concordance_sigle','text_id',
       'left_context','first_match','second_match','third_match','right_context']
    returns one big LaTeX string:

    \\begin{table}
    K46 spiegel\_5130368
    \\begin{tabularx}{\\textwidth}{QQQ}
    … & … & …
    \\end{tabularx}
    . \\newline
    K47 welt\_5130368
    \\begin{tabularx}{\\textwidth}{QQQ}
    … & … & …
    \\end{tabularx}
    \\caption{…}
    \\end{table}
    """
    lines = []
    lines.append(r"\begin{table}")

    for i, r in enumerate(results):
        # escape underscores in text_id
        text_id = r['text_id'].replace("__", "_")
        # build the 3‐part match cell with \newline
        match_cell = (
            f"{r['first_match']}\\newline\n"
            f"{r['second_match']}\\newline\n"
            f"{r['third_match']}"
        )
        # header for this concordance
        lines.append(f"{r['concordance_sigle']} {text_id}")
        # the tabularx block
        lines.append(r"\begin{tabularx}{\textwidth}{QQQ}")
        lines.append(f"{r['left_context']} & {match_cell} & {r['right_context']} \\\\")
        lines.append(r"\end{tabularx}")

        # if not the last concordance, add the “. \newline” separator
        if i < len(results) - 1:
            lines.append(r"\bigskip \newline")

    #lines.append(f"\\caption{{{caption}}}")
    lines.append(r"\end{table}")

    return "\n".join(lines)


def comment_concordances(text):
    # Regular expression to find sections between \begin{longtable} and \end{longtable}
    longtable_pattern = re.compile(
        r"(\\begin{longtable}.*?\\end{longtable})", re.DOTALL
    )

    # Regular expression to check if the section contains 'K01', 'K02', ..., 'K99'
    K_pattern = re.compile(r"K\d{2}")

    # Function to add '% ' at the beginning of each line in a section
    def comment_section(section):
        lines = section.splitlines()
        commented_lines = ["% " + line for line in lines]
        return "\n".join(commented_lines)

    # Iterate over each matched longtable section
    def process_section(match):
        section = match.group(0)
        # Check if the section contains 'K01', 'K02', ..., 'K99'
        if K_pattern.search(section):
            # If it does, comment the section
            # Note: You might want to update this list if other K-numbers also need special commenting.
            if "K01" in section:
                latex_output = """
\\begin{table}
K01 taz\_2208663
\\begin{tabularx}{\\textwidth}{QQQ}
man den Rechten keine Plattform geben darf in den Stuben der Verfassungsschutzämter wird lieber geschreddert & \\textbf{Nazis} & sind Soziopathen wer sich ihnen offen entgegenstellt braucht Mut und verdient jede Unterstützung ganz am \\\\
\end{tabularx}
\end{table}
"""
                return latex_output
            else:
                parsed_section = extract_longtable_rows(section)
                if not parsed_section: # If no rows were extracted (e.g. due to structure mismatch)
                    # Decide how to handle: comment out, return original, or log error
                    # Returning original section if parsing fails for some reason
                    # print(f"Warning: Could not parse section starting with {section[:100]}")
                    return section # Or comment_section(section) if preferred
                latex_output = make_grouped_table(parsed_section, caption="xxx") # Consider dynamic caption
                return latex_output
            
        # Otherwise, return the section unchanged
        return section

    # Replace each matching section with the processed version
    commented_text = longtable_pattern.sub(process_section, text)

    return commented_text



In [8]:
import re


def transform_latex_figures(text):
    """
    Transforms LaTeX figure blocks from the input format to the desired format.

    Args:
        text (str): The input LaTeX text containing figure blocks.

    Returns:
        str: The transformed LaTeX text with figure environments.
    """

    # Pattern to match the includegraphics and caption block
    figure_pattern = re.compile(
        r"% \\includegraphics\[width=([^\]]+),height=([^\]]+)\]\{([^}]+)\}\s*\n"  # Match includegraphics
        r"% \\phantomsection\\label\{([^}]+)\}\{\}(.*?)\n",  # Match label and caption text
        re.DOTALL,
    )

    def replace_figure(match):
        # Extract width, height, image path, label, and caption text
        width = match.group(1)  # Width of the image
        height = match.group(2)  # Height of the image
        image_path = match.group(3).replace(".emf", ".png")  # Path to the image
        label = match.group(4)  # Label for the figure
        caption_text = match.group(5).strip().split("Abbildung : ")[1]  # Caption text

        # Change the image path to "figures" folder and use a generic name for the figure
        new_image_path = (
            "figures/media/" + image_path.split("/")[-1]
        )  # Use only the filename part

        # Create label text, converting the label to "fig:" prefix
        new_label = f"fig:{label.split('_')[-1]}"

        # Generate the replacement text block
        transformed_block = (
            r"\begin{figure}" + "\n"
            r"  \includegraphics[width=\textwidth]{" + new_image_path + "}\n"
            r"  \caption{" + f"\\label{{{new_label}}}" + caption_text + "}\n"
            r"\end{figure}" + "\n"
        )

        return transformed_block

    # Apply the transformation to all matches of the pattern
    transformed_text = re.sub(figure_pattern, replace_figure, text)

    return transformed_text

import re

def format_clusters(text):
    # """
    # Surrounds all clusters in the given text with the \cluster{} LaTeX command,
    # including handling of \textbar as a separator and proper backslash matching.
    # """
    # # Updated regex to allow matching \textbar and properly handle LaTeX syntax
    # pattern = re.compile(
    #     r'\b(?:sp|we|ta|99|02|05|08|11|14|17|20)\\_(?:[A-Za-zÄÖÜäöüß0-9-]|\\_)+(?:(\\textbar (?:[A-Za-zÄÖÜäöüß0-9-]|\\_)+))*\b'
    # )

    # def replacer(match):
    #     matched_text = match.group(0)
    #     matched_text = matched_text.replace(r"\textbar ", r"{\breakbar}")
    #     # Wrap the match in \cluster{}
    #     return f'\\cluster{{{matched_text}}}'

    # # Replace all matches in the text
    # result = pattern.sub(replacer, text)
    result = text.replace(r"\textbar ", r"\textbar").replace(r" \textbar", r"\textbar").replace(r"\textbar{}", r"\textbar")
    result = result.replace(r"\textbar", r"{\breakbar}")
    result = result.replace(r"\underline{", r"\cluster{")
    result = result.replace(r"\cluster{frame}", r"\underline{frame}") # except one real usage of underline

    return result



In [9]:
num_title ={
    "1": "EINLEITUNG",
    "2": "THEORIE",
    "2.1": "DISKURSLINGUISTIK",
    "2.1.1": "VERORTUNG IN DER LINGUISTIK",
    "2.1.2": "DISKURSBEGRIFF UND METHODOLOGISCHE ZUGÄNGE ZUM DISKURS",
    "2.1.3": "FRAMES ALS ELEMENTE DISKURSIVEN WISSENS",
    "2.2": "FRAME-SEMANTIK",
    "2.2.1": "FILLMORES FRAME-SEMANTIK",
    "2.2.1.1": "Entailment, Case Grammar, Understanding Semantics",
    "2.2.1.2": "FrameNet",
    "2.2.1.2.1": "Struktur eines Frame-Eintrags",
    "2.2.1.2.2": "Erstellungsprozess eines Frame-Eintrags",
    "2.2.1.2.3": "Welche Frames werden aufgenommen?",
    "2.2.2": "DAS FRAME-KONZEPT DIETRICH BUSSES",
    "2.2.2.1": "Überblick",
    "2.2.2.2": "Type- und Token-Frames",
    "2.2.2.3": "Frame-Struktur: Slots, Filler und Relationen",
    "2.2.2.4": "Sprachoberflächliche Bezugspunkte",
    "2.2.2.5": "Perspektivierung, Kontext und Frame-Wandel",
    "2.2.2.6": "Frames und Kognition",
    "2.3": "FRAMING",
    "2.3.1": "FRAMING IN DEN KOMMUNIKATIONSWISSENSCHAFTEN BEI ENTMAN",
    "2.3.2": "FRAMING IN DER LINGUISTISCHEN FRAME-THEORIE",
    "2.4": "DISTRIBUTIONELLE SEMANTIK, KORPUSPRAGMATIK UND DAS CORPUS-DRIVEN PARADIGMA",
    "2.4.1": "DISTRIBUTIONELLE SEMANTIK",
    "2.4.2": "KORPUSPRAGMATIK",
    "2.4.3": "SPRACHGEBRAUCHSMUSTER",
    "2.4.4": "DER CORPUS-DRIVEN ANSATZ",
    "2.5": "EXTREMISMUS IN DEN SOZIALWISSENSCHAFTEN",
    "2.5.1": "DEFINITORISCHE GRUNDLAGEN DER EXTREMISMUSTHEORIE",
    "2.5.2": "DEFINITIONEN EINZELNER EXTREMISMUSVARIANTEN",
    "2.5.3": "BEGRIFFLICHE VERWIRRUNGEN UND KRITIK AM EXTREMISMUS-KONZEPT",
    "2.6": "FORSCHUNGSSTAND",
    "2.6.1": "EXTREMISMUS-DISKURS",
    "2.6.1.1": "Sozialwissenschaftlich-kritische Diskursanalyse bei Ackermann et al. (2015)",
    "2.6.1.2": "Weitere Arbeiten zum Extremismusdiskurs und -begriff",
    "2.6.2": "FRAME-INDUKTION",
    "2.6.3": "DISTRIBUTIONELL-SEMANTISCHE METHODIK",
    "2.7": "RESÜMEE",
    "2.7.1": "WARUM FRAMES?",
    "2.7.2": "DISTRIBUTIONELL-SEMANTISCHE MODELLIERBARKEIT VON FRAMES",
    "2.7.3": "INHALTLICHE MINDESTANFORDERUNGEN AN EINEN EXTREMISMUS-FRAME",
    "3": "DATEN UND METHODIK",
    "3.1": "KORPUS",
    "3.1.1": "AUSWAHL DER ZEITUNGEN UND DES UNTERSUCHUNGSZEITRAUMS",
    "3.1.2": "VERWENDETE KORPUS-SOFTWARE",
    "3.1.3": "KENNDATEN ZUM KORPUS UND KORPUSAUFBEREITUNG",
    "3.2": "METHODIK UND OPERATIONALISIERUNG",
    "3.2.1": "PROZESS DER METHODENENTWICKLUNG",
    "3.2.2": "BEGRIFFSKLÄRUNGEN: QUANTITATIV UND QUALITATIV – CORPUS-DRIVEN UND CORPUS-BASED SOWIE DIE ROLLE VON HERMENEUTIK",
    "3.2.3": "METHODEN DER DISTRIBUTIONELLEN SEMANTIK: WORD EMBEDDINGS UND KOLLOKATIONEN",
    "3.2.3.1": "Kollokationen als syntagmatische distributionelle Muster",
    "3.2.3.1.1": "Was sind Kollokationen?",
    "3.2.3.1.2": "Wie funktioniert die Berechnung von Kollokationen?",
    "3.2.3.2": "Word Embeddings als paradigmatische distributionelle Muster",
    "3.2.3.2.1": "Was sind Word Embeddings?",
    "3.2.3.2.2": "Wie funktioniert die Berechnung von Word Embeddings?",
    "3.2.3.2.3": "Vergleichbarkeit von Word-Embedding-Modellen",
    "3.2.3.2.4": "Clustering von Word Embeddings",
    "3.2.4": "OPERATIONALISIERUNG VON FRAMES FÜR DISTRIBUTIONELLE METHODEN",
    "3.2.4.1": "Geclusterte Word Embeddings als Frame-Elemente",
    "3.2.4.2": "Nähe im Vektorraum als Frame-Evokationspotenzial und Perspektivierung",
    "3.2.4.3": "Kollokation von Word-Embedding-Clustern als Frame-Relationen und Perspektivierung, Kollokationsnetzwerke als Frames",
    "3.2.4.4": "Netzwerk-Communities als thematische Teil-Frames",
    "3.2.4.5": "Berechnung von Frames auf Basis unterschiedlicher Subkorpora als Modellierung von Kontexten und Frame-Wandel",
    "3.2.5": "TECHNISCH-METHODISCHE UMSETZUNG: VOM ROHEN TEXT ZUM GRAPHEN",
    "3.2.5.1": "Generelles Vorgehen",
    "3.2.5.2": "Details der Korpusaufbereitung",
    "3.2.5.3": "Berechnung und Clustering der Word-Embedding-Modelle",
    "3.2.5.4": "Berechnung der Kollokationen",
    "3.2.5.5": "Berechnung des Extremismus-Centroid und Begrenzung der Knotenmenge",
    "3.2.5.6": "Visualisierung als Netzwerk-Graph und Erkennung von Communities",
    "3.2.6": "WIE DER GRAPH ZU LESEN IST – UND MÖGLICHE FALLSTRICKE",
    "3.2.6.1": "Überblick und Bedienung",
    "3.2.6.2": "Interpretation der Knoten",
    "3.2.6.3": "Interpretation der Kanten",
    "3.2.6.4": "Interpretation der räumlichen Struktur",
    "3.2.6.5": "Mögliche Fallstricke bei der Interpretation",
    "4": "ANALYSE",
    "4.1": "VORGEHEN UND ZIELSETZUNG",
    "4.2": "MAKROSTRUKTUR: THEMATISCHE TEIL-FRAMES",
    "4.3": "MESOSTRUKTUR: EXTREMISMUSVARIANTEN-FRAMES",
    "4.3.1": "DER ZEITRAUM 1999 – 2001",
    "4.3.1.1": "Corpus-driven Voranalyse: Cluster mit hoher Cosinus-Distanz im Vergleich zum Gesamtkorpus",
    "4.3.1.2": "Rechts- und Linksextremismus",
    "4.3.1.3": "Islamismus",
    "4.3.1.4": "Fazit",
    "4.3.2": "DER ZEITRAUM 2002 – 2004",
    "4.3.2.1": "Corpus-driven Voranalyse: Thematisch gruppierte Auswahl semantisch äquivalenter Cluster im Vergleich zu den Kern-FE des vorigen Zeitraums",
    "4.3.2.2": "Corpus-driven Voranalyse: Cluster mit hoher Cosinus-Distanz im Vergleich zum vorigen Zeitraum",
    "4.3.2.3": "Rechts- und Linksextremismus",
    "4.3.2.4": "Islamismus",
    "4.3.2.5": "Fazit",
    "4.3.3": "DER ZEITRAUM 2005 – 2007",
    "4.3.3.1": "Corpus-driven Voranalyse: Thematisch gruppierte Auswahl semantisch äquivalenter Cluster im Vergleich zu den Kern-FE des vorigen Zeitraums",
    "4.3.3.2": "Corpus-driven Voranalyse: Cluster mit hoher Cosinus-Distanz im Vergleich zum vorigen Zeitraum",
    "4.3.3.3": "Rechts- und Linksextremismus",
    "4.3.3.4": "Islamismus",
    "4.3.3.5": "Fazit",
    "4.3.4": "DER ZEITRAUM 2008 – 2010",
    "4.3.4.1": "Corpus-driven Voranalyse: Thematisch gruppierte Auswahl semantisch äquivalenter Cluster im Vergleich zu den Kern-FE des vorigen Zeitraums",
    "4.3.4.2": "Corpus-driven Voranalyse: Cluster mit hoher Cosinus-Distanz im Vergleich zum vorigen Zeitraum",
    "4.3.4.3": "Rechts- und Linksextremismus",
    "4.3.4.4": "Islamismus",
    "4.3.4.5": "Fazit",
    "4.3.5": "DER ZEITRAUM 2011 – 2013",
    "4.3.5.1": "Corpus-driven Voranalyse: Thematisch gruppierte Auswahl semantisch äquivalenter Cluster im Vergleich zu den Kern-FE des vorigen Zeitraums",
    "4.3.5.2": "Corpus-driven Voranalyse: Cluster mit hoher Cosinus-Distanz im Vergleich zum vorigen Zeitraum",
    "4.3.5.3": "Rechts- und Linksextremismus",
    "4.3.5.4": "Islamismus",
    "4.3.5.5": "Fazit",
    "4.3.6": "DER ZEITRAUM 2014 – 2016",
    "4.3.6.1": "Corpus-driven Voranalyse: Thematisch gruppierte Auswahl semantisch äquivalenter Cluster im Vergleich zu den Kern-FE des vorigen Zeitraums",
    "4.3.6.2": "Corpus-driven Voranalyse: Cluster mit hoher Cosinus-Distanz im Vergleich zum vorigen Zeitraum",
    "4.3.6.3": "Rechts- und Linksextremismus",
    "4.3.6.4": "Islamismus",
    "4.3.6.5": "Fazit",
    "4.3.7": "DER ZEITRAUM 2017 – 2019",
    "4.3.7.1": "Corpus-driven Voranalyse: Thematisch gruppierte Auswahl semantisch äquivalenter Cluster im Vergleich zu den Kern-FE des vorigen Zeitraums",
    "4.3.7.2": "Corpus-driven Voranalyse: Cluster mit hoher Cosinus-Distanz im Vergleich zum vorigen Zeitraum",
    "4.3.7.3": "Rechts- und Linksextremismus",
    "4.3.7.4": "Islamismus",
    "4.3.7.5": "Fazit",
    "4.3.8": "DER ZEITRAUM 2020 – AUGUST 2021",
    "4.3.8.1": "Corpus-driven Voranalyse: Thematisch gruppierte Auswahl semantisch äquivalenter Cluster im Vergleich zu den Kern-FE des vorigen Zeitraums",
    "4.3.8.2": "Corpus-driven Voranalyse: Cluster mit hoher Cosinus-Distanz im Vergleich zum vorigen Zeitraum",
    "4.3.8.3": "Rechts- und Linksextremismus",
    "4.3.8.4": "Islamismus",
    "4.3.8.5": "Fazit",
    "4.3.9": "TAZ",
    "4.3.9.1": "Corpus-driven Voranalyse: Thematisch gruppierte Auswahl semantisch äquivalenter Cluster im Vergleich zu den Kern-FE der diachronen Analyse",
    "4.3.9.2": "Corpus-driven Voranalyse: Cluster mit hoher Cosinus-Distanz im Vergleich zum Gesamtkorpus",
    "4.3.9.3": "Rechts- und Linksextremismus",
    "4.3.9.4": "Islamismus",
    "4.3.10": "SPIEGEL",
    "4.3.10.1": "Corpus-driven Voranalyse: Thematisch gruppierte Auswahl semantisch äquivalenter Cluster im Vergleich zu den Kern-FE der diachronen Analyse",
    "4.3.10.2": "Corpus-driven Voranalyse: Cluster mit hoher Cosinus-Distanz im Vergleich zum Gesamtkorpus",
    "4.3.10.3": "Rechts- und Linksextremismus",
    "4.3.10.4": "Islamismus",
    "4.3.11": "WELT",
    "4.3.11.1": "Corpus-driven Voranalyse: Thematisch gruppierte Auswahl semantisch äquivalenter Cluster im Vergleich zu den Kern-FE der diachronen Analyse",
    "4.3.11.2": "Corpus-driven Voranalyse: Cluster mit hoher Cosinus-Distanz im Vergleich zum Gesamtkorpus",
    "4.3.11.3": "Rechts- und Linksextremismus",
    "4.3.11.4": "Islamismus",
    "4.4": "FAZIT DER ANALYSE",
    "4.4.1": "VARIANTENÜBERGREIFENDE KONSTANTEN DES FRAMINGS VON EXTREMISMUS",
    "4.4.2": "POLITISCHER EXTREMISMUS",
    "4.4.3": "ISLAMISMUS",
    "5": "DISKUSSION, LIMITATIONEN, PERSPEKTIVEN UND FAZIT",
    "5.1": "DISKUSSION",
    "5.2": "METHODISCHE LIMITATIONEN UND PERSPEKTIVEN",
    "5.3": "FAZIT"
}

In [10]:
chapters = split_sections(output_citekeys)

for section_name, content in chapters.items():

    ### langscipress-specific changes ###
    # renaming of sections
    chapters[section_name] = (
        content.replace("\section", "\chapter")
        .replace("subsection", "section")
        .replace("\paragraph", "\subsubsection")
        .replace("\subparagraph", "\subsubsubsection")
    )
    # correcting citation
    chapters[section_name] = chapters[section_name].replace(
        "[siehe auch][aus kognitionswissenschaftler Perspektive]",
        "[siehe auch aus kognitionswissenschaftler Perspektive]",
    )
    # Format Frames and Frame-Elements
    chapters[section_name] = chapters[section_name].replace(
        "\emph{commercial-event}-Frame",
        "\FN{Commercial\_event}-Frame"
        ).replace(
        "Gathering\_up-Frame",
        "\FN{Gathering\_up}-Frame"
        ).replace(
            "{KÄUFERS}", "{käufers}").replace("{WARE}", "{ware}").replace("{KÄUFER}", "{käufer}").replace("HÄNDLERS", r"\textsc{händlers}").replace("{GELDES}", "{geldes}"
            ).replace("Research-Frame", "\FN{Research}-Frame").replace(
            "QUESTION, RESEARCHER, TOPIC",
            r"\textsc{question}, \textsc{researcher}, \textsc{topic}"
        ).replace("FE FIELD", r"FE \textsc{field}").replace("DURATION\_OF\_STATE, MANNER, MEANS, PLACE, POPULATION, PURPOSE, TIME und TYPE", r"\textsc{duration\_of\_state}, \textsc{manner}, \textsc{means}, \textsc{place}, \textsc{population}, \textsc{purpose}, \textsc{time} und \textsc{type}").replace(
            ", QUESTION und TOPIC", r", \textsc{question} und \textsc{topic}"
        ).replace(
            "Scrutiny-Frame", "\FN{Scrutiny}-Frame"
        ).replace("Trying_out-Frame", "\FN{Trying\_out}-Frame").replace("Cogitation-Frame", "\FN{Cogitation}-Frame").replace(
            "Bail\_decision-, der Knot\_creation\_scenario- oder der Hunting\_success\_or\_failure-Frame", "\FN{Bail\_decision}-, der \FN{Knot\_creation\_scenario}- oder der \FN{Hunting\_success\_or\_-failure}-Frame"
        ).replace("Ingestion-Frame", "\FN{Ingestion}-Frame").replace("wie Terrorism statt", "wie \FN{Terrorism} statt").replace("Hostile\_encounter", "\FN{Hostile\_encounter}"). replace("""Catastrophe-Frame, der als „Undesirable\_event which affects the Patient negatively""", r"""\FN{Catastrophe}-Frame, der als „\textsc{undesirable\_event} which affects the \textsc{patient} negatively""")

    ### graphics & concordances ###
    chapters[section_name] = (
        chapters[section_name]
        .replace(
            """
\includegraphics[width=6.29583in,height=4.64861in]{./figures/media/image17.png}\includegraphics[width=4.36047in,height=4.36496in]{./figures/media/image18.png}

\phantomsection\label{_Ref162016499}{}Abbildung : Extremismus-Frame des Zeitraumes 2020 -- 2021.

\phantomsection\label{_Ref162016524}{}Abbildung : Ausschnitt aus dem Politik-Teil-Frame des Zeitraums 2020 -- 2021.
""",
            """
\includegraphics[width=6.29583in,height=4.64861in]{media/image18.png}

\phantomsection\label{_Ref162016499}{}Abbildung : Extremismus-Frame des Zeitraumes 2020 -- 2021.

\includegraphics[width=4.36047in,height=4.36496in]{media/image17.png}

\phantomsection\label{_Ref162016524}{}Abbildung : Ausschnitt aus dem Politik-Teil-Frame des Zeitraums 2020 -- 2021.
""",
        )
        .replace(
            r"""\includegraphics[width=6.29514in,height=4.10347in]{./figures/media/image26.png}\includegraphics[width=6.15764in,height=4.40417in]{./figures/media/image27.png}

\phantomsection\label{_Ref162439766}{}Abbildung : Ausschnitt aus dem thematischen Teil-Frame ‚Außenpolitik` am Beispiel des Clusters 14\ul{\_Reformen}.
""",
            r"""\includegraphics[width=6.29514in,height=4.10347in]{media/image26.png}

\phantomsection\label{_Ref162439766}{}Abbildung : Ausschnitt aus dem thematischen Teil-Frame ‚Außenpolitik` am Beispiel des Clusters \underline{14\_Reformen}.

\includegraphics[width=6.15764in,height=4.40417in]{media/image27.png}
""",
        ).replace(
            r"""\includegraphics[width=6.29583in,height=3.02569in]{./figures/media/image7.png}(4) We have acquired more than 100 works. (GETTING)""",
            r"""(4) We have acquired more than 100 works. (GETTING)

\includegraphics[width=6.29583in,height=3.02569in]{./figures/media/image7.png}
"""
        )
    )
    chapters[section_name] = (
        chapters[section_name]
        .replace("\includegraphics", "% \includegraphics")
        .replace("\phantomsection", "% \phantomsection")
    )  # commenting out graphics and phantomsection to deal with them later
    chapters[section_name] = transform_latex_figures(
        chapters[section_name]
    )  # transforming figures
    chapters[section_name] = chapters[section_name].replace(
        "K16\n\nspiegel\_1333818", "\\begin{minipage}[t]{\linewidth}\\raggedright\nK16\\\\\n\nspiegel\_\\\\\n1333818\strut\n\end{minipage} "
    ).replace("K15\n\nwelt", "K15\\\\\n\nwelt") 
    chapters[section_name] = comment_concordances(
        chapters[section_name]
    )  # commenting out concordances to deal with them later

    ### conversion errors ###

    chapters[section_name] = chapters[section_name].replace(
        "(Bubenhofer 2022: 205--206)",
        r"\autocite[205-206]{bubenhoferExplorationSemantischerRaeume2022}",
    )  # correcting not converted citation
    chapters[section_name] = chapters[section_name].replace(
        "(Baker et al. 2020: 4)", r"\autocite[4]{bakerChangingFramesObesity2020}"
    )  # correcting not converted citation
    chapters[section_name] = chapters[section_name].replace(
        "(Kozlowski, Taddy \& Evans 2019: 914--915)",
        r"\autocite[914-915]{kozlowskiGeometryCultureAnalyzing2019}",
    )  # correcting not converted citation
    chapters[section_name] = chapters[section_name].replace(
        r"\ul{", r"\underline{"
    )  # correcting underlining
    chapters[section_name] = (
        chapters[section_name].replace("ﬀ", "ff").replace("ﬁ", "fi").replace("ﬂ", "fl")
    )  # correcting ligatures
    chapters[section_name] = chapters[section_name].replace(
        r"\textquotesingle ", "'"
    ).replace(
        r"\textquotesingle{}", "`"
    )  # correcting apostrophes
    chapters[section_name] = chapters[section_name].replace(
        " ]", "]"
    )  # removing spaces before closing brackets
    chapters[section_name] = chapters[section_name].replace(
        "siehe ausführlicher Kapitel xxx", "siehe ausführlicher Kapitel 3.2.2"
    )  # fixing chapter reference

    # fix tables
    for table_fix in tables.values():
        chapters[section_name] = chapters[section_name].replace(
            table_fix[0], table_fix[1]
        )

    # ### own errors ###

    chapters[section_name] = chapters[section_name].replace(
        r'innewohnenden "Zauber"',
        r"innewohnenden „Zauber``",
    )  # correcting zotero's quotation marks

    chapters[section_name] = chapters[section_name].replace(
        "widerspiegel.",
        "widerspiegelt.",
    ).replace(
        "anderer umgebender Wörter",
        "anderer sie umgebender Wörter",
    ).replace(
        ", und insofern ein Fall von",
        " und insofern ein Fall von"
    )
    
    chapters[section_name] = chapters[section_name].replace(
        r"So wäre es wenig sinnvoll, in Bezug auf die Jugendorganisation der AfD, Kollokationen",
        r"So wäre es in Bezug auf die Jugendorganisation der AfD wenig sinnvoll, Kollokationen",
    )
    ### formatting
    chapters[section_name] = format_clusters(chapters[section_name])

    ### fix section

    chapters[section_name] = chapters[section_name].replace(
        r"""\section{\texorpdfstring{\hfill\break
Distributionelle Semantik, Korpuspragmatik und das \emph{corpus-driven} Paradigma}{ Distributionelle Semantik, Korpuspragmatik und das corpus-driven Paradigma}}\label{distributionelle-semantik-korpuspragmatik-und-das-corpus-driven-paradigma}
""",
        r"\section{Distributionelle Semantik, Korpuspragmatik und das \emph{corpus-driven} Paradigma}\label{distributionelle-semantik-korpuspragmatik-und-das-corpus-driven-paradigma}"
        )





In [11]:
import re

def collect_sectioning_lines(text_or_file, from_file=False):
    """
    Collect all LaTeX sectioning lines (chapter, section, subsection, etc.)
    Returns the full lines, not just the command names.
    """
    # Updated pattern - removed capturing group or use non-capturing group
    pattern = r'^\\(?:part|chapter|section|subsection|subsubsection|subsubsubsection|paragraph|subparagraph)\{.*$'
    
    if from_file:
        with open(text_or_file, 'r', encoding='utf-8') as file:
            content = file.read()
    else:
        content = text_or_file
    
    # This will now return the full matching lines
    return re.findall(pattern, content, re.MULTILINE)

# Alternative using finditer (more explicit)
def collect_sectioning_lines_v2(text_or_file, from_file=False):
    """Using finditer to get full matches"""
    pattern = r'^\\(part|chapter|section|subsection|subsubsection|subsubsubsection|paragraph|subparagraph)\{.*$'
    
    if from_file:
        with open(text_or_file, 'r', encoding='utf-8') as file:
            content = file.read()
    else:
        content = text_or_file
    
    return [match.group(0) for match in re.finditer(pattern, content, re.MULTILINE)]


In [12]:
chapters.keys()

dict_keys(['inhaltsverzeichnis', 'einleitung', 'theorie', 'methodik', 'analyse', 'diskussion', 'verzeichnisse', 'anhang'])

In [13]:
sections = []

for chap, content in chapters.items():
    if chap in ["inhaltsverzeichnis", "verzeichnisse", "anhang"]:
        continue
    # Collect sectioning lines from each chapter
    section_lines = collect_sectioning_lines(content)
    # Add chapter name to each line for context
    #section_lines = [f"{chap}: {line}" for line in section_lines]
    sections.extend(section_lines)

In [14]:
sections[:4]

['\\chapter{Einleitung}\\label{einleitung}',
 '\\chapter{Theorie}\\label{theorie}',
 '\\section{Diskurslinguistik}\\label{diskurslinguistik}',
 '\\subsection{Verortung in der Linguistik}\\label{verortung-in-der-linguistik}']

In [15]:
import pandas as pd

df = pd.DataFrame({
    "numeration": num_title.keys(),
    "title": num_title.values(),
    "lines": sections,
})
df.head(40)

,numeration,title,lines
0,1,EINLEITUNG,\chapter{Einleitung}\label{einleitung}
1,2,THEORIE,\chapter{Theorie}\label{theorie}
2,2.1,DISKURSLINGUISTIK,\section{Diskurslinguistik}\label{diskurslingu...
3,2.1.1,VERORTUNG IN DER LINGUISTIK,\subsection{Verortung in der Linguistik}\label...
4,2.1.2,DISKURSBEGRIFF UND METHODOLOGISCHE ZUGÄNGE ZUM...,\subsection{Diskursbegriff und methodologische...
5,2.1.3,FRAMES ALS ELEMENTE DISKURSIVEN WISSENS,\subsection{Frames als Elemente diskursiven Wi...
6,2.2,FRAME-SEMANTIK,\section{Frame-Semantik}\label{frame-semantik}
7,2.2.1,FILLMORES FRAME-SEMANTIK,\subsection{Fillmores Frame-Semantik}\label{fi...
8,2.2.1.1,"Entailment, Case Grammar, Understanding Semantics","\subsubsection{Entailment, Case Grammar, Under..."
9,2.2.1.2,FrameNet,\subsubsection{\texorpdfstring{\emph{FrameNet}...


In [16]:
import re

def parse_latex_line(line):
    """Parse a single LaTeX line to extract section name and label"""
    # Extract section name
    section_match = re.search(r'\\[^{]*\{([^}]*)\}', line)
    section_name = section_match.group(1) if section_match else None
    
    # Extract label
    label_match = re.search(r'\\label\{([^}]*)\}', line)
    label = label_match.group(1) if label_match else None
    
    return pd.Series([section_name, label])

# Apply to your dataframe
df[['section_name', 'label']] = df['lines'].apply(parse_latex_line)


In [17]:
df.lines[23]

'\\section{Distributionelle Semantik, Korpuspragmatik und das \\emph{corpus-driven} Paradigma}\\label{distributionelle-semantik-korpuspragmatik-und-das-corpus-driven-paradigma}'

In [18]:
list(df.label)

['einleitung',
 'theorie',
 'diskurslinguistik',
 'verortung-in-der-linguistik',
 'diskursbegriff-und-methodologische-zuguxe4nge-zum-diskurs',
 'frames-als-elemente-diskursiven-wissens',
 'frame-semantik',
 'fillmores-frame-semantik',
 'entailment-case-grammar-understanding-semantics',
 'framenet',
 'struktur-eines-frame-eintrags',
 'erstellungsprozess-eines-frame-eintrags',
 'welche-frames-werden-aufgenommen',
 'das-frame-konzept-dietrich-busses',
 'uxfcberblick',
 'type--und-token-frames',
 'frame-struktur-slots-filler-und-relationen',
 'sprachoberfluxe4chliche-bezugspunkte',
 'perspektivierung-kontext-und-frame-wandel',
 'frames-und-kognition',
 'framing',
 'framing-in-den-kommunikationswissenschaften-bei-entman',
 'framing-in-der-linguistischen-frame-theorie',
 'distributionelle-semantik-korpuspragmatik-und-das-corpus-driven-paradigma',
 'distributionelle-semantik',
 'korpuspragmatik',
 'sprachgebrauchsmuster',
 'der-corpus-driven-ansatz',
 'extremismus-in-den-sozialwissenschaften'

In [19]:
df["new_label"] = df["label"].str.replace("uxfc", "ue").str.replace("uxf6", "oe").str.replace("uxfc", "ue").str.replace("uxdf", "ss").str.replace("uxe4", "ae").str.replace("uxc4", "Ae").str.replace("uxd6", "Oe").str.replace("uxdc", "Ue")

In [20]:
# create a column new_line with the label replaced by new_label
df["new_line"] = df.apply(
    lambda row: row["lines"].replace(row["label"], row["new_label"]), axis=1
)

In [21]:
import re
import pandas as pd


def transform_kapitel_references(text, numeration_to_label):
    """
    Transform "Kapitel X" mentions into LaTeX references
    """
    # Regex pattern to match "Kapitel" followed by numbers and dots
    pattern = r'Kapitel (\d+(?:\.\d+)*)'
    
    def replace_kapitel(match):
        numeration = match.group(1)
        if numeration in numeration_to_label:
            label = numeration_to_label[numeration]
            return f'Kapitel~\\ref{{{label}}}'  # Using ~ for non-breaking space
        else:
            return match.group(0)  # Return original if no mapping found
    
    return re.sub(pattern, replace_kapitel, text)

# Create the mapping
numeration_to_label = dict(zip(df['numeration'].astype(str), df['new_label']))


In [22]:
pattern = r'\\caption(?:\[[^\]]*\])?\{.*?\\label\{(fig:[^}]+)\}.*?\}'

matches = re.findall(pattern, " ".join(chapters.values()), flags=re.DOTALL)

matches_without_duplicates = []

for match in matches:
    if match not in matches_without_duplicates:
        matches_without_duplicates.append(match)

new_labels =  [
    "fig:framenet-research",
    "fig:framegrapher-research",
    "fig:maus-frame",
    "fig:merkel-frame",
    "fig:hufeisen",
    "fig:bert",
    "fig:kozlowski",
    "fig:som-tod",
    "fig:medien-ideologie-verortung",
    "fig:subkorpora-umfang",
    "fig:staedte-loglikelihood-cluster",
    "fig:cbow-architektur",
    "fig:kmeans-beispiel-k2",
    "fig:tiere-embeddings-semantik",
    "fig:pca-nachbarn-extremistische",
    "fig:extremismus-frame-2020-21",
    "fig:politik-frame-2020-21",
    "fig:hanau-frame-2020-21",
    "fig:rechtsextremismus-frame-1999-2001",
    "fig:cluster-tooltip-nsu-mundlos",
    "fig:politik-beispiel",
    "fig:krieg-beispiel",
    "fig:straftaten-beispiel",
    "fig:staat-beispiel",
    "fig:aussenpolitik-beispiel",
    "fig:demonstrationen-beispiel",
    "fig:flucht-beispiel",
    "fig:npd-frame",
    "fig:perspektivierung-nazis",
    "fig:perspektivierung-neonazis",
    "fig:xenophobie-rassismus",
    "fig:pds-vernetzung",
    "fig:islamismus-demonstrationen",
    "fig:silvesternacht-straftaten",
    "fig:george-floyd-frame"
]

fig_old_new = dict(zip(matches_without_duplicates, new_labels))

In [23]:
for section_name, content in chapters.items():
    # Replace section labels in the chapter content
    content = content.replace(
        "Kapiteln 3.2.4.3 und 4.3.1.2", f"Kapiteln~\\ref{{{numeration_to_label['3.2.4.3']}}} und \\ref{{{numeration_to_label['4.3.1.2']}}}"
        ). replace(
            "Kapiteln 3.1.3 und 3.2.5", f"Kapiteln~\\ref{{{numeration_to_label['3.1.3']}}} und \\ref{{{numeration_to_label['3.2.5']}}}"
        ).replace(
            "Kapiteln 2.6.3 und 3.2.3.2", f"Kapiteln~\\ref{{{numeration_to_label['2.6.3']}}} und \\ref{{{numeration_to_label['3.2.3.2']}}}"
        ).replace(
            "Kapitel 2.4.1, 2.6.3 und 3.2.3.1", f"Kapitel~\\ref{{{numeration_to_label['2.4.1']}}}, \\ref{{{numeration_to_label['2.6.3']}}} und \\ref{{{numeration_to_label['3.2.3.1']}}}"
        ).replace(
            "(Kapitel 3.2.4.4, 3.2.5.6)", f"(Kapitel~\\ref{{{numeration_to_label['3.2.4.4']}}} und \\ref{{{numeration_to_label['3.2.5.6']}}})"
        ).replace(
            "Kapitel 2.5 und 2.6.1", f"Kapitel~\\ref{{{numeration_to_label['2.5']}}} und \\ref{{{numeration_to_label['2.6.1']}}}"
        ).replace(
            "siehe Kapitel 3.1.3 und 3.2.5.2 zur", f"siehe Kapitel~\\ref{{{numeration_to_label['3.1.3']}}} und \\ref{{{numeration_to_label['3.2.5.2']}}} zur"
        ).replace(
            "Kapitel 2.6.1 bzw. 2.7.3", f"Kapitel~\\ref{{{numeration_to_label['2.6.1']}}} bzw. \\ref{{{numeration_to_label['2.7.3']}}}"
        ).replace(
            "Kapitel 2.5.3 und 2.6.1", f"Kapitel~\\ref{{{numeration_to_label['2.5.3']}}} und \\ref{{{numeration_to_label['2.6.1']}}}"
        ).replace(
            "siehe Kapitel 2.5.3 -- 2.6.1.2", f"siehe Kapitel~\\ref{{{numeration_to_label['2.5.3']}}} –- \\ref{{{numeration_to_label['2.6.1.2']}}}"
        ).replace(
            "Kapitel (3.2.5)", f"Kapitel~\\ref{{{numeration_to_label['3.2.5']}}}"
        ).replace(
            "Kapitel (3.2.6)", f"Kapitel~\\ref{{{numeration_to_label['3.2.6']}}}"
        ).replace(
            "Kapitel (2.4)", f"Kapitel~\\ref{{{numeration_to_label['2.4']}}}"
        ).replace(
            "(siehe Kapitel 2.2.1.2 und 2.7.1)", f"(siehe Kapitel~\\ref{{{numeration_to_label['2.2.1.2']}}} und \\ref{{{numeration_to_label['2.7.1']}}})"
        ).replace(
            "(Kapitel 2.2.1.2.3 und 2.7.1)", f"(Kapitel~\\ref{{{numeration_to_label['2.2.1.2.3']}}} und \\ref{{{numeration_to_label['2.7.1']}}})"
        ).replace(
            "(siehe ausführlicher Kapitel 2.4.1 und 3.2.3.2)", f"(siehe ausführlicher Kapitel~\\ref{{{numeration_to_label['2.4.1']}}} und \\ref{{{numeration_to_label['3.2.3.2']}}})"
        ).replace(
            "der Cluster Kapitel 3.2.3.2.4 und 3.2.5.3)", f"der Cluster Kapitel~\\ref{{{numeration_to_label['3.2.3.2.4']}}} und \\ref{{{numeration_to_label['3.2.5.3']}}})"
        ).replace(
            "(Kapitel 3.2.1 und 3.2.2)", f"(Kapitel~\\ref{{{numeration_to_label['3.2.1']}}} und \\ref{{{numeration_to_label['3.2.2']}}})"
        )
    for _, row in df.iterrows():
        if row['new_label']:
            content = content.replace(f"\\label{{{row['label']}}}", f"\\label{{{row['new_label']}}}")
            content = transform_kapitel_references(content, numeration_to_label)
    for old_label, new_label in fig_old_new.items():
        content = content.replace(f"\\label{{{old_label}}}", f"\\label{{{new_label}}}")
    chapters[section_name] = content

In [24]:
df

,numeration,title,lines,section_name,label,new_label,new_line
0,1,EINLEITUNG,\chapter{Einleitung}\label{einleitung},Einleitung,einleitung,einleitung,\chapter{Einleitung}\label{einleitung}
1,2,THEORIE,\chapter{Theorie}\label{theorie},Theorie,theorie,theorie,\chapter{Theorie}\label{theorie}
2,2.1,DISKURSLINGUISTIK,\section{Diskurslinguistik}\label{diskurslingu...,Diskurslinguistik,diskurslinguistik,diskurslinguistik,\section{Diskurslinguistik}\label{diskurslingu...
3,2.1.1,VERORTUNG IN DER LINGUISTIK,\subsection{Verortung in der Linguistik}\label...,Verortung in der Linguistik,verortung-in-der-linguistik,verortung-in-der-linguistik,\subsection{Verortung in der Linguistik}\label...
4,2.1.2,DISKURSBEGRIFF UND METHODOLOGISCHE ZUGÄNGE ZUM...,\subsection{Diskursbegriff und methodologische...,Diskursbegriff und methodologische Zugänge zum...,diskursbegriff-und-methodologische-zuguxe4nge-...,diskursbegriff-und-methodologische-zugaenge-zu...,\subsection{Diskursbegriff und methodologische...
...,...,...,...,...,...,...,...
147,4.4.3,ISLAMISMUS,\subsection{Islamismus}\label{islamismus-11},Islamismus,islamismus-11,islamismus-11,\subsection{Islamismus}\label{islamismus-11}
148,5,"DISKUSSION, LIMITATIONEN, PERSPEKTIVEN UND FAZIT","\chapter{Diskussion, Limitationen, Perspektive...","Diskussion, Limitationen, Perspektiven und Fazit",diskussion-limitationen-perspektiven-und-fazit,diskussion-limitationen-perspektiven-und-fazit,"\chapter{Diskussion, Limitationen, Perspektive..."
149,5.1,DISKUSSION,\section{Diskussion}\label{diskussion},Diskussion,diskussion,diskussion,\section{Diskussion}\label{diskussion}
150,5.2,METHODISCHE LIMITATIONEN UND PERSPEKTIVEN,\section{Methodische Limitationen und Perspekt...,Methodische Limitationen und Perspektiven,methodische-limitationen-und-perspektiven,methodische-limitationen-und-perspektiven,\section{Methodische Limitationen und Perspekt...


In [25]:
# Write the content of each chapter to a file
for chapter_name, content in chapters.items():
    file_path = "chapters/" + f"{chapter_name}.tex"
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(content)

In [31]:
dict(zip([i + 1 for i in range(len(new_labels))], new_labels))

{1: 'fig:framenet-research',
 2: 'fig:framegrapher-research',
 3: 'fig:maus-frame',
 4: 'fig:merkel-frame',
 5: 'fig:hufeisen',
 6: 'fig:bert',
 7: 'fig:kozlowski',
 8: 'fig:som-tod',
 9: 'fig:medien-ideologie-verortung',
 10: 'fig:subkorpora-umfang',
 11: 'fig:staedte-loglikelihood-cluster',
 12: 'fig:cbow-architektur',
 13: 'fig:kmeans-beispiel-k2',
 14: 'fig:tiere-embeddings-semantik',
 15: 'fig:pca-nachbarn-extremistische',
 16: 'fig:extremismus-frame-2020-21',
 17: 'fig:politik-frame-2020-21',
 18: 'fig:hanau-frame-2020-21',
 19: 'fig:rechtsextremismus-frame-1999-2001',
 20: 'fig:cluster-tooltip-nsu-mundlos',
 21: 'fig:politik-beispiel',
 22: 'fig:krieg-beispiel',
 23: 'fig:straftaten-beispiel',
 24: 'fig:staat-beispiel',
 25: 'fig:aussenpolitik-beispiel',
 26: 'fig:demonstrationen-beispiel',
 27: 'fig:flucht-beispiel',
 28: 'fig:npd-frame',
 29: 'fig:perspektivierung-nazis',
 30: 'fig:perspektivierung-neonazis',
 31: 'fig:xenophobie-rassismus',
 32: 'fig:pds-vernetzung',
 33: 'fi